In [2]:
# ============================================================
# SECTION 1 FIX — LOCATE TEST LABELS CORRECTLY
# ============================================================

from pathlib import Path

print("=" * 70)
print("LOCATING TEST LABELS")
print("=" * 70)

# Your images are:
# G:\EcoBotX_YOLO\images\test

TEST_IMAGES = Path(r"G:\EcoBotX_YOLO\images\test")

# Correct YOLO label location:
TEST_LABELS = Path(r"G:\EcoBotX_YOLO\labels\test")

print(f"\nTest images:")
print(f"  {TEST_IMAGES}")

print(f"\nTest labels:")
print(f"  {TEST_LABELS}")


# ============================================================
# CHECK DIRECTORIES
# ============================================================

if not TEST_IMAGES.exists():
    raise FileNotFoundError(
        f"[ERROR] Test images directory not found:\n{TEST_IMAGES}"
    )

if not TEST_LABELS.exists():
    raise FileNotFoundError(
        f"[ERROR] Test labels directory not found:\n{TEST_LABELS}"
    )


# ============================================================
# COUNT FILES
# ============================================================

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}

test_image_files = sorted(
    [
        p for p in TEST_IMAGES.rglob("*")
        if p.is_file()
        and p.suffix.lower() in IMAGE_EXTENSIONS
    ]
)

test_label_files = sorted(
    TEST_LABELS.glob("*.txt")
)

print("\n" + "=" * 70)
print("DATASET CHECK")
print("=" * 70)

print(f"Test images : {len(test_image_files)}")
print(f"Test labels : {len(test_label_files)}")


# ============================================================
# VERIFY EXPECTED TEST SET
# ============================================================

assert len(test_image_files) == 1115, (
    f"Expected 1115 test images, found {len(test_image_files)}"
)

assert len(test_label_files) > 0, (
    "No test label files found."
)


print("\n[OK] Test images located successfully.")
print("[OK] Test labels located successfully.")


# ============================================================
# CHECK IMAGE/LABEL MATCHING
# ============================================================

image_stems = {p.stem for p in test_image_files}
label_stems = {p.stem for p in test_label_files}

missing_labels = image_stems - label_stems
extra_labels = label_stems - image_stems

print("\n" + "=" * 70)
print("IMAGE ↔ LABEL MATCHING")
print("=" * 70)

print(f"Images without labels : {len(missing_labels)}")
print(f"Labels without images : {len(extra_labels)}")

if missing_labels:
    print("\n[WARNING] Some images do not have labels.")
    print("Examples:")
    for name in sorted(missing_labels)[:10]:
        print(f"  {name}")

if extra_labels:
    print("\n[WARNING] Some labels do not have matching images.")
    print("Examples:")
    for name in sorted(extra_labels)[:10]:
        print(f"  {name}")


# ============================================================
# FINAL
# ============================================================

if not missing_labels and not extra_labels:
    print("\n[OK] Every test image has a matching label.")
else:
    print("\n[WARNING] Image/label mismatch detected.")

print("\n" + "=" * 70)
print("SECTION 1 LABEL LOCATION — COMPLETED")
print("=" * 70)

LOCATING TEST LABELS

Test images:
  G:\EcoBotX_YOLO\images\test

Test labels:
  G:\EcoBotX_YOLO\labels\test

DATASET CHECK
Test images : 1115
Test labels : 1115

[OK] Test images located successfully.
[OK] Test labels located successfully.

IMAGE ↔ LABEL MATCHING
Images without labels : 0
Labels without images : 0

[OK] Every test image has a matching label.

SECTION 1 LABEL LOCATION — COMPLETED


In [3]:

# ======================================================================
# EXPERIMENT 2 — CAN FAILURE ANALYSIS
# SECTION 2 — GROUND-TRUTH CAN ANALYSIS
# ======================================================================

from pathlib import Path
from collections import Counter
import yaml
import numpy as np

print("=" * 70)
print("EXPERIMENT 2 — CAN FAILURE ANALYSIS")
print("SECTION 2 — GROUND-TRUTH CAN ANALYSIS")
print("=" * 70)


# ======================================================================
# 1. PATHS
# ======================================================================

DATASET_YAML = Path(r"G:\EcoBotX_YOLO\dataset.yaml")
TEST_IMAGES = Path(r"G:\EcoBotX_YOLO\images\test")
TEST_LABELS = Path(r"G:\EcoBotX_YOLO\labels\test")

CAN_CLASS_ID = 1

print("\n[1] Checking paths...")

assert DATASET_YAML.exists(), f"Dataset YAML not found: {DATASET_YAML}"
assert TEST_IMAGES.exists(), f"Test image directory not found: {TEST_IMAGES}"
assert TEST_LABELS.exists(), f"Test label directory not found: {TEST_LABELS}"

print("[OK] Dataset YAML found")
print("[OK] Test images found")
print("[OK] Test labels found")


# ======================================================================
# 2. LOAD DATASET CONFIGURATION
# ======================================================================

with open(DATASET_YAML, "r", encoding="utf-8") as f:
    data_cfg = yaml.safe_load(f)

names = data_cfg["names"]

if isinstance(names, dict):
    class_names = {int(k): v for k, v in names.items()}
else:
    class_names = {i: v for i, v in enumerate(names)}

print("\n" + "=" * 70)
print("DATASET CLASSES")
print("=" * 70)

for class_id, class_name in class_names.items():
    print(f"  {class_id}: {class_name}")


# ======================================================================
# 3. FIND TEST IMAGES
# ======================================================================

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

image_files = sorted(
    [
        p for p in TEST_IMAGES.iterdir()
        if p.is_file() and p.suffix.lower() in image_extensions
    ]
)

print("\n" + "=" * 70)
print("TEST DATASET")
print("=" * 70)

print(f"Test images: {len(image_files)}")
print(f"Test labels: {len(list(TEST_LABELS.glob('*.txt')))}")


# ======================================================================
# 4. READ CAN GROUND-TRUTH BOXES
# ======================================================================
#
# YOLO label format:
#
# class_id  x_center  y_center  width  height
#
# Coordinates are normalized to [0, 1].
#
# We only analyze:
#
#     class_id == 1  → CAN
#
# ======================================================================

can_annotations = []
can_images = []

total_annotations = 0
total_can_annotations = 0

class_counter = Counter()

missing_labels = []
invalid_labels = []

for image_path in image_files:

    label_path = TEST_LABELS / f"{image_path.stem}.txt"

    if not label_path.exists():
        missing_labels.append(image_path.name)
        continue

    image_has_can = False

    try:
        with open(label_path, "r", encoding="utf-8") as f:
            lines = [line.strip() for line in f if line.strip()]

        for line_number, line in enumerate(lines, start=1):

            parts = line.split()

            if len(parts) != 5:
                invalid_labels.append(
                    (image_path.name, line_number, line)
                )
                continue

            class_id = int(float(parts[0]))
            x_center = float(parts[1])
            y_center = float(parts[2])
            width = float(parts[3])
            height = float(parts[4])

            class_counter[class_id] += 1
            total_annotations += 1

            if class_id != CAN_CLASS_ID:
                continue

            # ----------------------------------------------------------
            # CAN annotation
            # ----------------------------------------------------------

            total_can_annotations += 1
            image_has_can = True

            area = width * height
            aspect_ratio = width / height if height > 0 else 0

            can_annotations.append({
                "image": image_path.name,
                "width": width,
                "height": height,
                "area": area,
                "aspect_ratio": aspect_ratio,
                "x_center": x_center,
                "y_center": y_center
            })

    except Exception as e:
        invalid_labels.append(
            (image_path.name, "ERROR", str(e))
        )

    if image_has_can:
        can_images.append(image_path.name)


# ======================================================================
# 5. BASIC CAN STATISTICS
# ======================================================================

print("\n" + "=" * 70)
print("CAN GROUND-TRUTH SUMMARY")
print("=" * 70)

print(f"Total test images              : {len(image_files)}")
print(f"Total valid annotations        : {total_annotations}")
print(f"Total CAN annotations          : {total_can_annotations}")
print(f"Images containing CAN          : {len(can_images)}")
print(f"Images without CAN             : {len(image_files) - len(can_images)}")

print("\nClass distribution in test labels:")

for class_id in sorted(class_counter):
    name = class_names.get(class_id, f"CLASS_{class_id}")
    print(
        f"  {name:<10}: {class_counter[class_id]}"
    )


# ======================================================================
# 6. CAN BOX SIZE STATISTICS
# ======================================================================

if total_can_annotations == 0:
    raise RuntimeError(
        "No CAN annotations were found. "
        "Check CAN_CLASS_ID and the label files."
    )

widths = np.array([x["width"] for x in can_annotations])
heights = np.array([x["height"] for x in can_annotations])
areas = np.array([x["area"] for x in can_annotations])
aspect_ratios = np.array(
    [x["aspect_ratio"] for x in can_annotations]
)


def print_stats(name, values):
    print(f"\n{name}")
    print(f"  Minimum : {np.min(values):.4f}")
    print(f"  Mean    : {np.mean(values):.4f}")
    print(f"  Median  : {np.median(values):.4f}")
    print(f"  Maximum : {np.max(values):.4f}")
    print(f"  Std     : {np.std(values):.4f}")


print("\n" + "=" * 70)
print("CAN BOUNDING-BOX STATISTICS")
print("=" * 70)

print_stats("Normalized Width", widths)
print_stats("Normalized Height", heights)
print_stats("Normalized Area", areas)
print_stats("Aspect Ratio (Width / Height)", aspect_ratios)


# ======================================================================
# 7. CAN SIZE CATEGORIES
# ======================================================================
#
# Area is normalized because YOLO coordinates are normalized.
#
# Small:
#     area < 0.02
#
# Medium:
#     0.02 <= area < 0.10
#
# Large:
#     area >= 0.10
#
# These categories are used only for diagnostic analysis.
# They are NOT COCO size definitions.
# ======================================================================

small_mask = areas < 0.02
medium_mask = (areas >= 0.02) & (areas < 0.10)
large_mask = areas >= 0.10

small_count = int(np.sum(small_mask))
medium_count = int(np.sum(medium_mask))
large_count = int(np.sum(large_mask))

print("\n" + "=" * 70)
print("CAN SIZE DISTRIBUTION")
print("=" * 70)

print(
    f"Small CANs  (< 2% image area)  : "
    f"{small_count} "
    f"({small_count / len(areas) * 100:.2f}%)"
)

print(
    f"Medium CANs (2–10% area)       : "
    f"{medium_count} "
    f"({medium_count / len(areas) * 100:.2f}%)"
)

print(
    f"Large CANs  (>= 10% area)      : "
    f"{large_count} "
    f"({large_count / len(areas) * 100:.2f}%)"
)


# ======================================================================
# 8. APPROXIMATE PIXEL SIZE AT 640x640
# ======================================================================
#
# Your YOLO experiments use imgsz=640.
#
# Since normalized YOLO width/height are relative to the image,
# we can estimate the corresponding bounding-box dimensions:
#
#     pixel_width  = normalized_width  * 640
#     pixel_height = normalized_height * 640
#
# This helps determine whether CAN objects are becoming very small
# after resizing.
# ======================================================================

pixel_widths = widths * 640
pixel_heights = heights * 640

print("\n" + "=" * 70)
print("APPROXIMATE CAN BOX SIZE AT 640x640")
print("=" * 70)

print_stats("Estimated Pixel Width", pixel_widths)
print_stats("Estimated Pixel Height", pixel_heights)


# ======================================================================
# 9. VERY SMALL CAN DETECTION
# ======================================================================
#
# Diagnostic threshold:
#
#     width < 32 pixels OR height < 32 pixels
#
# This does not mean the object is impossible to detect.
# It simply identifies potentially difficult small-object examples.
# ======================================================================

very_small_mask = (pixel_widths < 32) | (pixel_heights < 32)

very_small_count = int(np.sum(very_small_mask))

print("\n" + "=" * 70)
print("VERY SMALL CAN ANALYSIS")
print("=" * 70)

print(
    f"CAN boxes with width < 32 px OR height < 32 px: "
    f"{very_small_count}"
)

print(
    f"Percentage of CAN annotations: "
    f"{very_small_count / len(areas) * 100:.2f}%"
)


# ======================================================================
# 10. EXTREME ASPECT RATIOS
# ======================================================================

wide_mask = aspect_ratios > 2.0
tall_mask = aspect_ratios < 0.5

wide_count = int(np.sum(wide_mask))
tall_count = int(np.sum(tall_mask))

print("\n" + "=" * 70)
print("CAN SHAPE ANALYSIS")
print("=" * 70)

print(
    f"Very wide CAN boxes (ratio > 2.0): "
    f"{wide_count}"
)

print(
    f"Very tall CAN boxes (ratio < 0.5): "
    f"{tall_count}"
)


# ======================================================================
# 11. LOCATION ANALYSIS
# ======================================================================

x_centers = np.array([x["x_center"] for x in can_annotations])
y_centers = np.array([x["y_center"] for x in can_annotations])

print("\n" + "=" * 70)
print("CAN POSITION ANALYSIS")
print("=" * 70)

print(f"Mean X center : {np.mean(x_centers):.4f}")
print(f"Mean Y center : {np.mean(y_centers):.4f}")

print(f"Min X center  : {np.min(x_centers):.4f}")
print(f"Max X center  : {np.max(x_centers):.4f}")

print(f"Min Y center  : {np.min(y_centers):.4f}")
print(f"Max Y center  : {np.max(y_centers):.4f}")


# ======================================================================
# 12. TOP 20 SMALLEST CAN OBJECTS
# ======================================================================

sorted_can = sorted(
    can_annotations,
    key=lambda x: x["area"]
)

print("\n" + "=" * 70)
print("20 SMALLEST CAN ANNOTATIONS")
print("=" * 70)

for i, item in enumerate(sorted_can[:20], start=1):

    print(
        f"{i:02d}. "
        f"{item['image']:<35} "
        f"W={item['width']:.4f} "
        f"H={item['height']:.4f} "
        f"Area={item['area']:.5f} "
        f"AR={item['aspect_ratio']:.2f}"
    )


# ======================================================================
# 13. TOP 20 LARGEST CAN OBJECTS
# ======================================================================

print("\n" + "=" * 70)
print("20 LARGEST CAN ANNOTATIONS")
print("=" * 70)

for i, item in enumerate(sorted_can[-20:][::-1], start=1):

    print(
        f"{i:02d}. "
        f"{item['image']:<35} "
        f"W={item['width']:.4f} "
        f"H={item['height']:.4f} "
        f"Area={item['area']:.5f} "
        f"AR={item['aspect_ratio']:.2f}"
    )


# ======================================================================
# 14. FINAL SECTION 2 SUMMARY
# ======================================================================

print("\n" + "=" * 70)
print("SECTION 2 — GROUND-TRUTH CAN ANALYSIS COMPLETED")
print("=" * 70)

print(f"""
CAN annotations       : {total_can_annotations}
Images containing CAN : {len(can_images)}

Mean CAN width        : {np.mean(widths):.4f}
Mean CAN height       : {np.mean(heights):.4f}
Mean CAN area         : {np.mean(areas):.5f}
Mean CAN aspect ratio : {np.mean(aspect_ratios):.3f}

Small CANs            : {small_count}
Medium CANs           : {medium_count}
Large CANs            : {large_count}

Very small CANs       : {very_small_count}
Very wide CANs        : {wide_count}
Very tall CANs        : {tall_count}
""")

print("[OK] SECTION 2 COMPLETED")
print("=" * 70)

EXPERIMENT 2 — CAN FAILURE ANALYSIS
SECTION 2 — GROUND-TRUTH CAN ANALYSIS

[1] Checking paths...
[OK] Dataset YAML found
[OK] Test images found
[OK] Test labels found

DATASET CLASSES
  0: BOTTLE
  1: CAN
  2: PAPER
  3: WRAPPER

TEST DATASET
Test images: 1115
Test labels: 1115

CAN GROUND-TRUTH SUMMARY
Total test images              : 1115
Total valid annotations        : 899
Total CAN annotations          : 228
Images containing CAN          : 228
Images without CAN             : 887

Class distribution in test labels:
  BOTTLE    : 217
  CAN       : 228
  PAPER     : 250
  WRAPPER   : 204

CAN BOUNDING-BOX STATISTICS

Normalized Width
  Minimum : 0.1562
  Mean    : 0.5165
  Median  : 0.5000
  Maximum : 1.0000
  Std     : 0.2209

Normalized Height
  Minimum : 0.1484
  Mean    : 0.4770
  Median  : 0.4453
  Maximum : 1.0000
  Std     : 0.1966

Normalized Area
  Minimum : 0.0295
  Mean    : 0.2686
  Median  : 0.2028
  Maximum : 0.8914
  Std     : 0.1893

Aspect Ratio (Width / Height)
  

In [5]:

# ======================================================================
# EXPERIMENT 2 — CAN FAILURE ANALYSIS
# SECTION 3 — EXPERIMENT 2 CAN PREDICTION ANALYSIS
# ======================================================================

from pathlib import Path
import yaml
import cv2
import numpy as np
from ultralytics import YOLO
from tqdm import tqdm
import shutil
import json

print("=" * 70)
print("EXPERIMENT 2 — CAN FAILURE ANALYSIS")
print("SECTION 3 — EXPERIMENT 2 CAN PREDICTION ANALYSIS")
print("=" * 70)


# ======================================================================
# 1. CONFIGURATION
# ======================================================================

DATASET_ROOT = Path(r"G:\EcoBotX_YOLO")

MODEL_PATH = Path(
    r"G:\EcoBotX_YOLO_training\experiment2_ecobotx_light-2\weights\best.pt"
)

TEST_IMAGE_DIR = DATASET_ROOT / "images" / "test"
TEST_LABEL_DIR = DATASET_ROOT / "labels" / "test"

OUTPUT_DIR = Path(
    r"G:\EcoBotX_YOLO_training\experiment2_can_failure_analysis"
)

FAILURE_DIR = OUTPUT_DIR / "failures"

MISSED_DIR = FAILURE_DIR / "missed_can"
LOW_CONF_DIR = FAILURE_DIR / "low_confidence_can"
WRONG_CLASS_DIR = FAILURE_DIR / "wrong_class_can"
CORRECT_DIR = FAILURE_DIR / "correct_can"

for folder in [
    OUTPUT_DIR,
    MISSED_DIR,
    LOW_CONF_DIR,
    WRONG_CLASS_DIR,
    CORRECT_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)


# ======================================================================
# 2. PARAMETERS
# ======================================================================

CAN_CLASS_ID = 1

# IoU threshold for deciding whether prediction matches CAN ground truth
IOU_THRESHOLD = 0.50

# Confidence threshold used for normal detection
CONF_THRESHOLD = 0.25

# Additional threshold for identifying weak CAN detections
LOW_CONF_THRESHOLD = 0.50


# ======================================================================
# 3. CHECK PATHS
# ======================================================================

print("\n[1] Checking paths...")

assert MODEL_PATH.exists(), f"Model not found: {MODEL_PATH}"
assert TEST_IMAGE_DIR.exists(), f"Test image directory not found: {TEST_IMAGE_DIR}"
assert TEST_LABEL_DIR.exists(), f"Test label directory not found: {TEST_LABEL_DIR}"

print("[OK] Experiment 2 model found")
print("[OK] Test image directory found")
print("[OK] Test label directory found")


# ======================================================================
# 4. LOAD MODEL
# ======================================================================

print("\n" + "=" * 70)
print("LOADING EXPERIMENT 2 MODEL")
print("=" * 70)

model = YOLO(str(MODEL_PATH))

print("[OK] Model loaded successfully")
print(f"Model: {MODEL_PATH}")

print("\nModel classes:")
for class_id, class_name in model.names.items():
    print(f"  {class_id}: {class_name}")


# ======================================================================
# 5. HELPER FUNCTIONS
# ======================================================================

def calculate_iou(box1, box2):
    """
    Calculate IoU between two boxes.

    Box format:
    [x1, y1, x2, y2]
    """

    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])

    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection_width = max(0, x2 - x1)
    intersection_height = max(0, y2 - y1)

    intersection_area = (
        intersection_width * intersection_height
    )

    area1 = max(0, box1[2] - box1[0]) * max(
        0, box1[3] - box1[1]
    )

    area2 = max(0, box2[2] - box2[0]) * max(
        0, box2[3] - box2[1]
    )

    union_area = area1 + area2 - intersection_area

    if union_area <= 0:
        return 0.0

    return intersection_area / union_area


def load_can_ground_truth(label_path, image_width, image_height):
    """
    Load CAN ground-truth boxes from YOLO label file.
    """

    can_boxes = []

    with open(label_path, "r") as f:
        lines = f.readlines()

    for line in lines:

        parts = line.strip().split()

        if len(parts) != 5:
            continue

        class_id = int(parts[0])

        if class_id != CAN_CLASS_ID:
            continue

        x_center = float(parts[1])
        y_center = float(parts[2])
        width = float(parts[3])
        height = float(parts[4])

        x1 = (x_center - width / 2) * image_width
        y1 = (y_center - height / 2) * image_height

        x2 = (x_center + width / 2) * image_width
        y2 = (y_center + height / 2) * image_height

        can_boxes.append(
            [x1, y1, x2, y2]
        )

    return can_boxes


# ======================================================================
# 6. ANALYSIS VARIABLES
# ======================================================================

total_can_gt = 0

true_positive_can = 0
missed_can = 0
low_conf_can = 0
wrong_class_can = 0

images_with_can = 0
images_with_missed_can = 0
images_with_wrong_class = 0
images_with_low_conf = 0

all_results = []

iou_values = []
confidence_values = []


# ======================================================================
# 7. GET TEST IMAGES
# ======================================================================

image_files = sorted(
    [
        p for p in TEST_IMAGE_DIR.iterdir()
        if p.suffix.lower() in [
            ".jpg",
            ".jpeg",
            ".png",
            ".bmp"
        ]
    ]
)

print("\n" + "=" * 70)
print("TEST DATASET")
print("=" * 70)

print(f"Test images: {len(image_files)}")


# ======================================================================
# 8. RUN EXPERIMENT 2
# ======================================================================

print("\n" + "=" * 70)
print("RUNNING EXPERIMENT 2 ON TEST SET")
print("=" * 70)

for image_path in tqdm(
    image_files,
    desc="Analyzing test images"
):

    label_path = TEST_LABEL_DIR / f"{image_path.stem}.txt"

    # --------------------------------------------------------------
    # Read image
    # --------------------------------------------------------------

    image = cv2.imread(str(image_path))

    if image is None:
        continue

    image_height, image_width = image.shape[:2]

    # --------------------------------------------------------------
    # Load CAN ground truth
    # --------------------------------------------------------------

    gt_can_boxes = load_can_ground_truth(
        label_path,
        image_width,
        image_height
    )

    if len(gt_can_boxes) == 0:
        continue

    images_with_can += 1
    total_can_gt += len(gt_can_boxes)

    # --------------------------------------------------------------
    # Run prediction
    # --------------------------------------------------------------

    results = model.predict(
        source=str(image_path),
        imgsz=640,
        conf=CONF_THRESHOLD,
        iou=0.7,
        verbose=False
    )

    result = results[0]

    predictions = []

    if result.boxes is not None:

        for box in result.boxes:

            xyxy = box.xyxy[0].cpu().numpy()

            cls = int(
                box.cls[0].cpu().item()
            )

            conf = float(
                box.conf[0].cpu().item()
            )

            predictions.append(
                {
                    "box": xyxy.tolist(),
                    "class_id": cls,
                    "confidence": conf
                }
            )

    # --------------------------------------------------------------
    # Match every CAN ground truth
    # --------------------------------------------------------------

    image_status = []

    for gt_index, gt_box in enumerate(gt_can_boxes):

        best_iou = 0.0
        best_prediction = None

        # Find prediction with highest IoU
        for pred in predictions:

            iou = calculate_iou(
                gt_box,
                pred["box"]
            )

            if iou > best_iou:
                best_iou = iou
                best_prediction = pred

        # ==========================================================
        # CASE 1 — CORRECT CAN DETECTION
        # ==========================================================

        if (
            best_prediction is not None
            and best_iou >= IOU_THRESHOLD
            and best_prediction["class_id"] == CAN_CLASS_ID
        ):

            true_positive_can += 1

            confidence_values.append(
                best_prediction["confidence"]
            )

            iou_values.append(best_iou)

            image_status.append("CORRECT")

            # Save only if confidence is relatively weak
            if best_prediction["confidence"] < LOW_CONF_THRESHOLD:

                low_conf_can += 1
                images_with_low_conf += 1

                shutil.copy2(
                    image_path,
                    LOW_CONF_DIR / image_path.name
                )

        # ==========================================================
        # CASE 2 — WRONG CLASS
        # ==========================================================

        elif (
            best_prediction is not None
            and best_iou >= IOU_THRESHOLD
            and best_prediction["class_id"] != CAN_CLASS_ID
        ):

            wrong_class_can += 1
            images_with_wrong_class += 1

            predicted_class = model.names[
                best_prediction["class_id"]
            ]

            image_status.append(
                f"WRONG_CLASS_{predicted_class}"
            )

            # Draw analysis image
            annotated = image.copy()

            # Ground truth CAN = green
            x1, y1, x2, y2 = map(
                int,
                gt_box
            )

            cv2.rectangle(
                annotated,
                (x1, y1),
                (x2, y2),
                (0, 255, 0),
                3
            )

            # Prediction = red
            px1, py1, px2, py2 = map(
                int,
                best_prediction["box"]
            )

            cv2.rectangle(
                annotated,
                (px1, py1),
                (px2, py2),
                (0, 0, 255),
                3
            )

            label = (
                f"GT: CAN | "
                f"Pred: {predicted_class} | "
                f"Conf: {best_prediction['confidence']:.2f} | "
                f"IoU: {best_iou:.2f}"
            )

            cv2.putText(
                annotated,
                label,
                (20, 35),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 0, 255),
                2
            )

            cv2.imwrite(
                str(
                    WRONG_CLASS_DIR /
                    image_path.name
                ),
                annotated
            )

        # ==========================================================
        # CASE 3 — MISSED CAN
        # ==========================================================

        else:

            missed_can += 1
            images_with_missed_can += 1

            image_status.append("MISSED")

            # Draw ground-truth box
            annotated = image.copy()

            x1, y1, x2, y2 = map(
                int,
                gt_box
            )

            cv2.rectangle(
                annotated,
                (x1, y1),
                (x2, y2),
                (0, 255, 0),
                3
            )

            label = (
                f"CAN MISSED | "
                f"Best IoU: {best_iou:.2f}"
            )

            cv2.putText(
                annotated,
                label,
                (20, 35),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 0, 255),
                2
            )

            cv2.imwrite(
                str(
                    MISSED_DIR /
                    image_path.name
                ),
                annotated
            )

    # --------------------------------------------------------------
    # Save image-level result
    # --------------------------------------------------------------

    all_results.append(
        {
            "image": image_path.name,
            "ground_truth_can": len(gt_can_boxes),
            "status": image_status
        }
    )


# ======================================================================
# 9. FINAL STATISTICS
# ======================================================================

print("\n" + "=" * 70)
print("SECTION 3 — CAN PREDICTION RESULTS")
print("=" * 70)

print(f"\nTotal CAN ground-truth boxes : {total_can_gt}")

print(f"Correct CAN detections       : {true_positive_can}")
print(f"Missed CAN boxes             : {missed_can}")
print(f"Wrong-class CAN boxes        : {wrong_class_can}")
print(f"Low-confidence CAN detections: {low_conf_can}")


# ======================================================================
# 10. CALCULATE CAN RECALL
# ======================================================================

if total_can_gt > 0:

    can_recall = (
        true_positive_can /
        total_can_gt
    )

else:

    can_recall = 0.0


print("\n" + "=" * 70)
print("CAN-SPECIFIC PERFORMANCE")
print("=" * 70)

print(
    f"CAN Recall: "
    f"{can_recall:.4f} "
    f"({can_recall * 100:.2f}%)"
)


# ======================================================================
# 11. IOU STATISTICS
# ======================================================================

if iou_values:

    print("\n" + "=" * 70)
    print("CAN LOCALIZATION QUALITY")
    print("=" * 70)

    print(
        f"Mean IoU   : {np.mean(iou_values):.4f}"
    )

    print(
        f"Median IoU : {np.median(iou_values):.4f}"
    )

    print(
        f"Minimum IoU: {np.min(iou_values):.4f}"
    )

    print(
        f"Maximum IoU: {np.max(iou_values):.4f}"
    )


# ======================================================================
# 12. CONFIDENCE STATISTICS
# ======================================================================

if confidence_values:

    print("\n" + "=" * 70)
    print("CAN CONFIDENCE ANALYSIS")
    print("=" * 70)

    print(
        f"Mean confidence   : "
        f"{np.mean(confidence_values):.4f}"
    )

    print(
        f"Median confidence : "
        f"{np.median(confidence_values):.4f}"
    )

    print(
        f"Minimum confidence: "
        f"{np.min(confidence_values):.4f}"
    )

    print(
        f"Maximum confidence: "
        f"{np.max(confidence_values):.4f}"
    )


# ======================================================================
# 13. SAVE JSON RESULTS
# ======================================================================

json_path = OUTPUT_DIR / "can_prediction_analysis.json"

summary = {
    "experiment": "Experiment 2",
    "model": str(MODEL_PATH),
    "test_images": len(image_files),
    "can_ground_truth": total_can_gt,
    "correct_can": true_positive_can,
    "missed_can": missed_can,
    "wrong_class_can": wrong_class_can,
    "low_confidence_can": low_conf_can,
    "can_recall": can_recall,
    "iou_mean": float(np.mean(iou_values))
        if iou_values else None,
    "iou_median": float(np.median(iou_values))
        if iou_values else None,
    "confidence_mean": float(np.mean(confidence_values))
        if confidence_values else None,
    "confidence_median": float(np.median(confidence_values))
        if confidence_values else None,
}

with open(
    json_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "summary": summary,
            "images": all_results
        },
        f,
        indent=4
    )


# ======================================================================
# 14. OUTPUT DIRECTORIES
# ======================================================================

print("\n" + "=" * 70)
print("FAILURE IMAGE OUTPUT")
print("=" * 70)

print(
    f"Missed CAN images       : "
    f"{MISSED_DIR}"
)

print(
    f"Wrong-class CAN images  : "
    f"{WRONG_CLASS_DIR}"
)

print(
    f"Low-confidence CAN      : "
    f"{LOW_CONF_DIR}"
)

print(
    f"Analysis JSON            : "
    f"{json_path}"
)


# ======================================================================
# 15. FINAL SUMMARY
# ======================================================================

print("\n" + "=" * 70)
print("SECTION 3 — COMPLETED")
print("=" * 70)

print(
    f"CAN Ground Truth : {total_can_gt}"
)

print(
    f"Correct          : {true_positive_can}"
)

print(
    f"Missed           : {missed_can}"
)

print(
    f"Wrong Class      : {wrong_class_can}"
)

print(
    f"Low Confidence   : {low_conf_can}"
)

print(
    f"CAN Recall       : {can_recall * 100:.2f}%"
)

print("\n[OK] SECTION 3 COMPLETED")


EXPERIMENT 2 — CAN FAILURE ANALYSIS
SECTION 3 — EXPERIMENT 2 CAN PREDICTION ANALYSIS

[1] Checking paths...
[OK] Experiment 2 model found
[OK] Test image directory found
[OK] Test label directory found

LOADING EXPERIMENT 2 MODEL
[OK] Model loaded successfully
Model: G:\EcoBotX_YOLO_training\experiment2_ecobotx_light-2\weights\best.pt

Model classes:
  0: BOTTLE
  1: CAN
  2: PAPER
  3: WRAPPER

TEST DATASET
Test images: 1115

RUNNING EXPERIMENT 2 ON TEST SET


Analyzing test images: 100%|██████████| 1115/1115 [00:04<00:00, 231.75it/s]


SECTION 3 — CAN PREDICTION RESULTS

Total CAN ground-truth boxes : 228
Correct CAN detections       : 196
Missed CAN boxes             : 22
Wrong-class CAN boxes        : 10
Low-confidence CAN detections: 16

CAN-SPECIFIC PERFORMANCE
CAN Recall: 0.8596 (85.96%)

CAN LOCALIZATION QUALITY
Mean IoU   : 0.8311
Median IoU : 0.8458
Minimum IoU: 0.5119
Maximum IoU: 0.9884

CAN CONFIDENCE ANALYSIS
Mean confidence   : 0.7575
Median confidence : 0.8032
Minimum confidence: 0.2517
Maximum confidence: 0.9498

FAILURE IMAGE OUTPUT
Missed CAN images       : G:\EcoBotX_YOLO_training\experiment2_can_failure_analysis\failures\missed_can
Wrong-class CAN images  : G:\EcoBotX_YOLO_training\experiment2_can_failure_analysis\failures\wrong_class_can
Low-confidence CAN      : G:\EcoBotX_YOLO_training\experiment2_can_failure_analysis\failures\low_confidence_can
Analysis JSON            : G:\EcoBotX_YOLO_training\experiment2_can_failure_analysis\can_prediction_analysis.json

SECTION 3 — COMPLETED
CAN Ground Tru

In [6]:

# ======================================================================
# EXPERIMENT 2 — CAN FAILURE ANALYSIS
# SECTION 4 — CAN FAILURE CHARACTERIZATION
# ======================================================================

from pathlib import Path
import cv2
import numpy as np
import json
from collections import Counter, defaultdict

print("=" * 70)
print("EXPERIMENT 2 — CAN FAILURE ANALYSIS")
print("SECTION 4 — CAN FAILURE CHARACTERIZATION")
print("=" * 70)


# ======================================================================
# 1. CONFIGURATION
# ======================================================================

DATASET_ROOT = Path(r"G:\EcoBotX_YOLO")

TEST_IMAGE_DIR = DATASET_ROOT / "images" / "test"
TEST_LABEL_DIR = DATASET_ROOT / "labels" / "test"

SECTION3_JSON = Path(
    r"G:\EcoBotX_YOLO_training\experiment2_can_failure_analysis"
    r"\can_prediction_analysis.json"
)

OUTPUT_DIR = Path(
    r"G:\EcoBotX_YOLO_training\experiment2_can_failure_analysis"
)

SECTION4_JSON = OUTPUT_DIR / "can_failure_characterization.json"

CAN_CLASS_ID = 1

CLASS_NAMES = {
    0: "BOTTLE",
    1: "CAN",
    2: "PAPER",
    3: "WRAPPER"
}


# ======================================================================
# 2. CHECK FILES
# ======================================================================

print("\n[1] Checking files...")

assert TEST_IMAGE_DIR.exists(), (
    f"Test image directory not found:\n{TEST_IMAGE_DIR}"
)

assert TEST_LABEL_DIR.exists(), (
    f"Test label directory not found:\n{TEST_LABEL_DIR}"
)

assert SECTION3_JSON.exists(), (
    f"Section 3 JSON not found:\n{SECTION3_JSON}"
)

print("[OK] Test image directory found")
print("[OK] Test label directory found")
print("[OK] Section 3 results found")


# ======================================================================
# 3. LOAD SECTION 3 RESULTS
# ======================================================================

print("\n[2] Loading Section 3 results...")

with open(
    SECTION3_JSON,
    "r",
    encoding="utf-8"
) as f:

    section3_data = json.load(f)

print("[OK] Section 3 results loaded")


# ======================================================================
# 4. BUILD FAILURE LOOKUP
# ======================================================================

print("\n[3] Building failure lookup...")

image_results = section3_data["images"]

failure_lookup = {}

for item in image_results:

    image_name = item["image"]

    statuses = item.get("status", [])

    failure_lookup[image_name] = statuses

print(
    f"[OK] Results loaded for "
    f"{len(failure_lookup)} images"
)


# ======================================================================
# 5. HELPER FUNCTIONS
# ======================================================================

def read_can_boxes(label_path):

    can_boxes = []

    with open(
        label_path,
        "r",
        encoding="utf-8"
    ) as f:

        lines = f.readlines()

    for line in lines:

        parts = line.strip().split()

        if len(parts) != 5:
            continue

        class_id = int(parts[0])

        if class_id != CAN_CLASS_ID:
            continue

        xc = float(parts[1])
        yc = float(parts[2])
        w = float(parts[3])
        h = float(parts[4])

        area = w * h

        aspect_ratio = (
            w / h
            if h > 0
            else 0
        )

        can_boxes.append(
            {
                "xc": xc,
                "yc": yc,
                "width": w,
                "height": h,
                "area": area,
                "aspect_ratio": aspect_ratio
            }
        )

    return can_boxes


def classify_size(area):

    if area < 0.02:
        return "SMALL"

    elif area < 0.10:
        return "MEDIUM"

    else:
        return "LARGE"


def classify_shape(aspect_ratio):

    if aspect_ratio > 2.0:
        return "VERY_WIDE"

    elif aspect_ratio < 0.5:
        return "VERY_TALL"

    else:
        return "NORMAL"


def classify_position(xc, yc):

    # Horizontal position
    if xc < 0.33:
        horizontal = "LEFT"

    elif xc > 0.67:
        horizontal = "RIGHT"

    else:
        horizontal = "CENTER"

    # Vertical position
    if yc < 0.33:
        vertical = "TOP"

    elif yc > 0.67:
        vertical = "BOTTOM"

    else:
        vertical = "CENTER"

    return horizontal, vertical


# ======================================================================
# 6. COLLECT CAN FAILURE DATA
# ======================================================================

print("\n" + "=" * 70)
print("ANALYZING CAN FAILURE CHARACTERISTICS")
print("=" * 70)


all_can = []
missed_can = []
wrong_class_can = []
correct_can = []

wrong_class_counter = Counter()


# ======================================================================
# 7. PROCESS EVERY TEST IMAGE
# ======================================================================

image_files = sorted(
    [
        p for p in TEST_IMAGE_DIR.iterdir()
        if p.suffix.lower() in [
            ".jpg",
            ".jpeg",
            ".png",
            ".bmp"
        ]
    ]
)


for image_path in image_files:

    image_name = image_path.name

    if image_name not in failure_lookup:
        continue

    label_path = TEST_LABEL_DIR / (
        image_path.stem + ".txt"
    )

    if not label_path.exists():
        continue

    can_boxes = read_can_boxes(label_path)

    if not can_boxes:
        continue

    statuses = failure_lookup[image_name]

    # --------------------------------------------------------------
    # Analyze each CAN annotation
    # --------------------------------------------------------------

    for index, can in enumerate(can_boxes):

        size_category = classify_size(
            can["area"]
        )

        shape_category = classify_shape(
            can["aspect_ratio"]
        )

        horizontal, vertical = classify_position(
            can["xc"],
            can["yc"]
        )

        record = {
            "image": image_name,
            "width": can["width"],
            "height": can["height"],
            "area": can["area"],
            "aspect_ratio": can["aspect_ratio"],
            "xc": can["xc"],
            "yc": can["yc"],
            "size": size_category,
            "shape": shape_category,
            "horizontal": horizontal,
            "vertical": vertical
        }

        all_can.append(record)

        # ----------------------------------------------------------
        # Determine status
        # ----------------------------------------------------------

        status = (
            statuses[index]
            if index < len(statuses)
            else "UNKNOWN"
        )

        if status == "CORRECT":
            correct_can.append(record)

        elif status == "MISSED":
            missed_can.append(record)

        elif status.startswith("WRONG_CLASS_"):

            wrong_class_can.append(record)

            predicted_class = (
                status.replace(
                    "WRONG_CLASS_",
                    ""
                )
            )

            record["predicted_class"] = (
                predicted_class
            )

            wrong_class_counter[
                predicted_class
            ] += 1


# ======================================================================
# 8. BASIC FAILURE COUNTS
# ======================================================================

print("\n" + "=" * 70)
print("CAN FAILURE COUNTS")
print("=" * 70)

print(
    f"Total CAN annotations : "
    f"{len(all_can)}"
)

print(
    f"Correct CAN           : "
    f"{len(correct_can)}"
)

print(
    f"Missed CAN            : "
    f"{len(missed_can)}"
)

print(
    f"Wrong-class CAN       : "
    f"{len(wrong_class_can)}"
)


# ======================================================================
# 9. FAILURE RATE
# ======================================================================

total_failures = (
    len(missed_can)
    + len(wrong_class_can)
)

if len(all_can) > 0:

    failure_rate = (
        total_failures /
        len(all_can)
    )

else:

    failure_rate = 0.0


print(
    f"\nOverall CAN failure rate : "
    f"{failure_rate * 100:.2f}%"
)


# ======================================================================
# 10. SIZE ANALYSIS
# ======================================================================

def distribution(records, key):

    counter = Counter(
        r[key]
        for r in records
    )

    return dict(counter)


print("\n" + "=" * 70)
print("CAN FAILURE BY OBJECT SIZE")
print("=" * 70)

size_all = distribution(
    all_can,
    "size"
)

size_missed = distribution(
    missed_can,
    "size"
)

size_wrong = distribution(
    wrong_class_can,
    "size"
)

for category in [
    "SMALL",
    "MEDIUM",
    "LARGE"
]:

    total = size_all.get(
        category,
        0
    )

    missed = size_missed.get(
        category,
        0
    )

    wrong = size_wrong.get(
        category,
        0
    )

    failures = missed + wrong

    rate = (
        failures / total * 100
        if total > 0
        else 0
    )

    print(
        f"{category:<8} "
        f"Total={total:<4} "
        f"Missed={missed:<4} "
        f"Wrong={wrong:<4} "
        f"Failure Rate={rate:.2f}%"
    )


# ======================================================================
# 11. ASPECT-RATIO ANALYSIS
# ======================================================================

print("\n" + "=" * 70)
print("CAN FAILURE BY SHAPE")
print("=" * 70)

shape_all = distribution(
    all_can,
    "shape"
)

shape_missed = distribution(
    missed_can,
    "shape"
)

shape_wrong = distribution(
    wrong_class_can,
    "shape"
)

for category in [
    "VERY_WIDE",
    "VERY_TALL",
    "NORMAL"
]:

    total = shape_all.get(
        category,
        0
    )

    missed = shape_missed.get(
        category,
        0
    )

    wrong = shape_wrong.get(
        category,
        0
    )

    failures = missed + wrong

    rate = (
        failures / total * 100
        if total > 0
        else 0
    )

    print(
        f"{category:<11} "
        f"Total={total:<4} "
        f"Missed={missed:<4} "
        f"Wrong={wrong:<4} "
        f"Failure Rate={rate:.2f}%"
    )


# ======================================================================
# 12. POSITION ANALYSIS — HORIZONTAL
# ======================================================================

print("\n" + "=" * 70)
print("CAN FAILURE BY HORIZONTAL POSITION")
print("=" * 70)

horizontal_all = distribution(
    all_can,
    "horizontal"
)

horizontal_missed = distribution(
    missed_can,
    "horizontal"
)

horizontal_wrong = distribution(
    wrong_class_can,
    "horizontal"
)

for category in [
    "LEFT",
    "CENTER",
    "RIGHT"
]:

    total = horizontal_all.get(
        category,
        0
    )

    missed = horizontal_missed.get(
        category,
        0
    )

    wrong = horizontal_wrong.get(
        category,
        0
    )

    failures = missed + wrong

    rate = (
        failures / total * 100
        if total > 0
        else 0
    )

    print(
        f"{category:<8} "
        f"Total={total:<4} "
        f"Missed={missed:<4} "
        f"Wrong={wrong:<4} "
        f"Failure Rate={rate:.2f}%"
    )


# ======================================================================
# 13. POSITION ANALYSIS — VERTICAL
# ======================================================================

print("\n" + "=" * 70)
print("CAN FAILURE BY VERTICAL POSITION")
print("=" * 70)

vertical_all = distribution(
    all_can,
    "vertical"
)

vertical_missed = distribution(
    missed_can,
    "vertical"
)

vertical_wrong = distribution(
    wrong_class_can,
    "vertical"
)

for category in [
    "TOP",
    "CENTER",
    "BOTTOM"
]:

    total = vertical_all.get(
        category,
        0
    )

    missed = vertical_missed.get(
        category,
        0
    )

    wrong = vertical_wrong.get(
        category,
        0
    )

    failures = missed + wrong

    rate = (
        failures / total * 100
        if total > 0
        else 0
    )

    print(
        f"{category:<8} "
        f"Total={total:<4} "
        f"Missed={missed:<4} "
        f"Wrong={wrong:<4} "
        f"Failure Rate={rate:.2f}%"
    )


# ======================================================================
# 14. NUMERICAL STATISTICS
# ======================================================================

def print_numeric_stats(
    name,
    records,
    key
):

    values = [
        r[key]
        for r in records
        if key in r
    ]

    if not values:
        return

    print(
        f"\n{name}"
    )

    print(
        f"  Mean   : {np.mean(values):.4f}"
    )

    print(
        f"  Median : {np.median(values):.4f}"
    )

    print(
        f"  Min    : {np.min(values):.4f}"
    )

    print(
        f"  Max    : {np.max(values):.4f}"
    )


print("\n" + "=" * 70)
print("NUMERICAL FAILURE CHARACTERISTICS")
print("=" * 70)

print_numeric_stats(
    "ALL CAN",
    all_can,
    "area"
)

print_numeric_stats(
    "MISSED CAN",
    missed_can,
    "area"
)

print_numeric_stats(
    "WRONG-CLASS CAN",
    wrong_class_can,
    "area"
)

print_numeric_stats(
    "ALL CAN ASPECT RATIO",
    all_can,
    "aspect_ratio"
)

print_numeric_stats(
    "MISSED CAN ASPECT RATIO",
    missed_can,
    "aspect_ratio"
)

print_numeric_stats(
    "WRONG-CLASS CAN ASPECT RATIO",
    wrong_class_can,
    "aspect_ratio"
)


# ======================================================================
# 15. WRONG CLASS CONFUSION ANALYSIS
# ======================================================================

print("\n" + "=" * 70)
print("CAN WRONG-CLASS CONFUSION")
print("=" * 70)

if wrong_class_counter:

    for class_name, count in (
        wrong_class_counter.most_common()
    ):

        percentage = (
            count /
            len(wrong_class_can) *
            100
        )

        print(
            f"CAN → {class_name:<8} "
            f"{count:>3} "
            f"({percentage:.2f}%)"
        )

else:

    print("No wrong-class CAN cases found.")


# ======================================================================
# 16. FAILURE CASES WITH THEIR PROPERTIES
# ======================================================================

print("\n" + "=" * 70)
print("MISSED CAN CASES")
print("=" * 70)

for i, record in enumerate(
    missed_can,
    start=1
):

    print(
        f"{i:02d}. "
        f"{record['image']} | "
        f"Area={record['area']:.4f} | "
        f"AR={record['aspect_ratio']:.2f} | "
        f"Center=({record['xc']:.2f}, "
        f"{record['yc']:.2f}) | "
        f"Size={record['size']} | "
        f"Shape={record['shape']}"
    )


# ======================================================================
# 17. WRONG CLASS CASES
# ======================================================================

print("\n" + "=" * 70)
print("WRONG-CLASS CAN CASES")
print("=" * 70)

for i, record in enumerate(
    wrong_class_can,
    start=1
):

    print(
        f"{i:02d}. "
        f"{record['image']} | "
        f"Predicted={record.get('predicted_class', 'UNKNOWN')} | "
        f"Area={record['area']:.4f} | "
        f"AR={record['aspect_ratio']:.2f} | "
        f"Center=({record['xc']:.2f}, "
        f"{record['yc']:.2f}) | "
        f"Size={record['size']} | "
        f"Shape={record['shape']}"
    )


# ======================================================================
# 18. SAVE RESULTS
# ======================================================================

output_data = {

    "total_can": len(all_can),

    "correct_can": len(correct_can),

    "missed_can": len(missed_can),

    "wrong_class_can": len(wrong_class_can),

    "failure_rate": failure_rate,

    "size": {
        "all": size_all,
        "missed": size_missed,
        "wrong_class": size_wrong
    },

    "shape": {
        "all": shape_all,
        "missed": shape_missed,
        "wrong_class": shape_wrong
    },

    "horizontal": {
        "all": horizontal_all,
        "missed": horizontal_missed,
        "wrong_class": horizontal_wrong
    },

    "vertical": {
        "all": vertical_all,
        "missed": vertical_missed,
        "wrong_class": vertical_wrong
    },

    "wrong_class_confusion": dict(
        wrong_class_counter
    ),

    "missed_cases": missed_can,

    "wrong_class_cases": wrong_class_can
}


with open(
    SECTION4_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output_data,
        f,
        indent=4
    )


# ======================================================================
# 19. FINAL SUMMARY
# ======================================================================

print("\n" + "=" * 70)
print("SECTION 4 — FINAL SUMMARY")
print("=" * 70)

print(
    f"Total CAN          : {len(all_can)}"
)

print(
    f"Correct            : {len(correct_can)}"
)

print(
    f"Missed             : {len(missed_can)}"
)

print(
    f"Wrong class        : {len(wrong_class_can)}"
)

print(
    f"Total failures     : {total_failures}"
)

print(
    f"Failure rate       : "
    f"{failure_rate * 100:.2f}%"
)

print("\nWrong-class confusion:")

if wrong_class_counter:

    for cls, count in wrong_class_counter.items():

        print(
            f"  CAN → {cls}: {count}"
        )

else:

    print("  None")


print("\nSaved characterization results:")
print(SECTION4_JSON)

print("\n" + "=" * 70)
print("[OK] SECTION 4 COMPLETED")
print("=" * 70)



EXPERIMENT 2 — CAN FAILURE ANALYSIS
SECTION 4 — CAN FAILURE CHARACTERIZATION

[1] Checking files...
[OK] Test image directory found
[OK] Test label directory found
[OK] Section 3 results found

[2] Loading Section 3 results...
[OK] Section 3 results loaded

[3] Building failure lookup...
[OK] Results loaded for 228 images

ANALYZING CAN FAILURE CHARACTERISTICS

CAN FAILURE COUNTS
Total CAN annotations : 228
Correct CAN           : 196
Missed CAN            : 22
Wrong-class CAN       : 10

Overall CAN failure rate : 14.04%

CAN FAILURE BY OBJECT SIZE
SMALL    Total=0    Missed=0    Wrong=0    Failure Rate=0.00%
MEDIUM   Total=44   Missed=3    Wrong=0    Failure Rate=6.82%
LARGE    Total=184  Missed=19   Wrong=10   Failure Rate=15.76%

CAN FAILURE BY SHAPE
VERY_WIDE   Total=21   Missed=1    Wrong=1    Failure Rate=9.52%
VERY_TALL   Total=13   Missed=1    Wrong=0    Failure Rate=7.69%
NORMAL      Total=194  Missed=20   Wrong=9    Failure Rate=14.95%

CAN FAILURE BY HORIZONTAL POSITION
LEF

In [7]:
# ======================================================================
# EXPERIMENT 2 — CAN FAILURE ANALYSIS
# SECTION 5 — VISUAL CAN FAILURE INSPECTION
# ======================================================================

from pathlib import Path
import json
import math
import cv2
import numpy as np
import matplotlib.pyplot as plt

print("=" * 70)
print("EXPERIMENT 2 — CAN FAILURE ANALYSIS")
print("SECTION 5 — VISUAL CAN FAILURE INSPECTION")
print("=" * 70)


# ======================================================================
# 1. CONFIGURATION
# ======================================================================

DATASET_ROOT = Path(r"G:\EcoBotX_YOLO")

TEST_IMAGES = DATASET_ROOT / "images" / "test"
TEST_LABELS = DATASET_ROOT / "labels" / "test"

SECTION4_JSON = Path(
    r"G:\EcoBotX_YOLO_training\experiment2_can_failure_analysis"
    r"\can_failure_characterization.json"
)

MODEL_PATH = Path(
    r"G:\EcoBotX_YOLO_training"
    r"\experiment2_ecobotx_light-2\weights\best.pt"
)

OUTPUT_ROOT = Path(
    r"G:\EcoBotX_YOLO_training"
    r"\experiment2_can_failure_analysis"
    r"\section5_visual_inspection"
)

MISSED_DIR = OUTPUT_ROOT / "missed_can"
WRONG_DIR = OUTPUT_ROOT / "wrong_class_can"

MISSED_DIR.mkdir(parents=True, exist_ok=True)
WRONG_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = {
    0: "BOTTLE",
    1: "CAN",
    2: "PAPER",
    3: "WRAPPER"
}

CAN_ID = 1

IOU_THRESHOLD = 0.50
CONF_THRESHOLD = 0.25

print("\n[1] Configuration")
print("-" * 70)

print(f"Model       : {MODEL_PATH}")
print(f"Test images : {TEST_IMAGES}")
print(f"Test labels : {TEST_LABELS}")
print(f"Section 4   : {SECTION4_JSON}")
print(f"Output      : {OUTPUT_ROOT}")


# ======================================================================
# 2. CHECK FILES
# ======================================================================

print("\n[2] Checking required files...")

assert MODEL_PATH.exists(), f"Model not found: {MODEL_PATH}"
assert TEST_IMAGES.exists(), f"Test image directory not found: {TEST_IMAGES}"
assert TEST_LABELS.exists(), f"Test label directory not found: {TEST_LABELS}"
assert SECTION4_JSON.exists(), f"Section 4 JSON not found: {SECTION4_JSON}"

print("[OK] Model found")
print("[OK] Test image directory found")
print("[OK] Test label directory found")
print("[OK] Section 4 characterization found")


# ======================================================================
# 3. LOAD MODEL
# ======================================================================

print("\n[3] Loading Experiment 2 model...")

from ultralytics import YOLO

model = YOLO(str(MODEL_PATH))

print("[OK] Model loaded")
print(f"Model classes: {model.names}")


# ======================================================================
# 4. LOAD SECTION 4 RESULTS
# ======================================================================

print("\n[4] Loading Section 4 results...")

with open(SECTION4_JSON, "r", encoding="utf-8") as f:
    characterization = json.load(f)

print("[OK] Section 4 results loaded")


# ======================================================================
# 5. EXTRACT FAILURE FILENAMES
# ======================================================================

print("\n[5] Extracting failure cases...")

missed_cases = []
wrong_cases = []

# Section 4 JSON structure can vary slightly depending on previous code.
# We search recursively for dictionaries containing filename/failure data.

def recursive_collect(obj):
    if isinstance(obj, dict):

        # Look for missed CAN information
        failure_type = str(
            obj.get("failure_type", obj.get("type", ""))
        ).lower()

        filename = (
            obj.get("filename")
            or obj.get("image")
            or obj.get("image_name")
            or obj.get("file")
        )

        predicted = (
            obj.get("predicted_class")
            or obj.get("predicted")
            or obj.get("prediction")
        )

        if filename:

            record = {
                "filename": str(filename),
                "predicted": predicted,
                "failure_type": failure_type
            }

            if "miss" in failure_type:
                missed_cases.append(record)

            elif "wrong" in failure_type:
                wrong_cases.append(record)

        for value in obj.values():
            recursive_collect(value)

    elif isinstance(obj, list):
        for item in obj:
            recursive_collect(item)


recursive_collect(characterization)

# Remove duplicates
def unique_cases(cases):
    seen = set()
    result = []

    for c in cases:
        key = c["filename"]

        if key not in seen:
            seen.add(key)
            result.append(c)

    return result


missed_cases = unique_cases(missed_cases)
wrong_cases = unique_cases(wrong_cases)

print(f"[INFO] Missed cases found      : {len(missed_cases)}")
print(f"[INFO] Wrong-class cases found : {len(wrong_cases)}")


# ======================================================================
# 6. FALLBACK — READ FAILURE DIRECTORIES
# ======================================================================

# If Section 4 JSON does not contain filenames in a machine-readable
# structure, use the folders produced during Section 3.

if len(missed_cases) == 0:

    print("\n[INFO] Searching missed CAN directory...")

    section3_missed = Path(
        r"G:\EcoBotX_YOLO_training"
        r"\experiment2_can_failure_analysis"
        r"\failures\missed_can"
    )

    if section3_missed.exists():

        for p in section3_missed.iterdir():

            if p.is_file():

                missed_cases.append({
                    "filename": p.name,
                    "predicted": None,
                    "failure_type": "missed"
                })

if len(wrong_cases) == 0:

    print("[INFO] Searching wrong-class CAN directory...")

    section3_wrong = Path(
        r"G:\EcoBotX_YOLO_training"
        r"\experiment2_can_failure_analysis"
        r"\failures\wrong_class_can"
    )

    if section3_wrong.exists():

        for p in section3_wrong.iterdir():

            if p.is_file():

                wrong_cases.append({
                    "filename": p.name,
                    "predicted": None,
                    "failure_type": "wrong_class"
                })


print("\nFailure cases available:")
print(f"  Missed CAN      : {len(missed_cases)}")
print(f"  Wrong-class CAN : {len(wrong_cases)}")


# ======================================================================
# 7. HELPER FUNCTIONS
# ======================================================================

def load_yolo_labels(label_path, image_width, image_height):

    boxes = []

    if not label_path.exists():
        return boxes

    with open(label_path, "r", encoding="utf-8") as f:

        for line in f:

            parts = line.strip().split()

            if len(parts) != 5:
                continue

            cls = int(parts[0])
            xc = float(parts[1])
            yc = float(parts[2])
            w = float(parts[3])
            h = float(parts[4])

            x1 = (xc - w / 2) * image_width
            y1 = (yc - h / 2) * image_height
            x2 = (xc + w / 2) * image_width
            y2 = (yc + h / 2) * image_height

            boxes.append({
                "class_id": cls,
                "class_name": CLASS_NAMES.get(cls, str(cls)),
                "xyxy": [x1, y1, x2, y2],
                "xc": xc,
                "yc": yc,
                "w": w,
                "h": h,
                "area": w * h,
                "aspect_ratio": w / h if h > 0 else 0
            })

    return boxes


def calculate_iou(box_a, box_b):

    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    iw = max(0, ix2 - ix1)
    ih = max(0, iy2 - iy1)

    intersection = iw * ih

    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)

    union = area_a + area_b - intersection

    if union <= 0:
        return 0.0

    return intersection / union


def find_image(filename):

    exact = TEST_IMAGES / filename

    if exact.exists():
        return exact

    matches = list(TEST_IMAGES.glob(filename))

    if matches:
        return matches[0]

    # Try matching by filename suffix
    for p in TEST_IMAGES.iterdir():

        if p.name == filename:
            return p

    return None


def find_label(image_path):

    return TEST_LABELS / f"{image_path.stem}.txt"


# ======================================================================
# 8. ANALYZE ONE FAILURE
# ======================================================================

def analyze_failure(case, failure_type):

    filename = case["filename"]

    image_path = find_image(filename)

    if image_path is None:

        print(f"[WARNING] Image not found: {filename}")
        return None

    image = cv2.imread(str(image_path))

    if image is None:

        print(f"[WARNING] Could not read: {image_path}")
        return None

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    height, width = image.shape[:2]

    label_path = find_label(image_path)

    ground_truth = load_yolo_labels(
        label_path,
        width,
        height
    )

    can_boxes = [
        x for x in ground_truth
        if x["class_id"] == CAN_ID
    ]

    if len(can_boxes) == 0:
        print(f"[WARNING] No CAN label found: {filename}")
        return None

    # Run model
    results = model.predict(
        source=str(image_path),
        imgsz=640,
        conf=CONF_THRESHOLD,
        verbose=False
    )

    result = results[0]

    predictions = []

    if result.boxes is not None:

        for i in range(len(result.boxes)):

            cls = int(
                result.boxes.cls[i].item()
            )

            conf = float(
                result.boxes.conf[i].item()
            )

            box = result.boxes.xyxy[i].cpu().numpy().tolist()

            predictions.append({
                "class_id": cls,
                "class_name": CLASS_NAMES.get(cls, str(cls)),
                "confidence": conf,
                "xyxy": box
            })

    # Find best prediction for each CAN
    for can in can_boxes:

        best_prediction = None
        best_iou = 0.0

        for pred in predictions:

            iou = calculate_iou(
                can["xyxy"],
                pred["xyxy"]
            )

            if iou > best_iou:

                best_iou = iou
                best_prediction = pred

        can["best_prediction"] = best_prediction
        can["best_iou"] = best_iou

    # Select the relevant CAN
    can = can_boxes[0]

    pred = can["best_prediction"]

    if pred is not None:

        pred_class = pred["class_name"]
        confidence = pred["confidence"]

    else:

        pred_class = "NO DETECTION"
        confidence = 0.0

    # Create visualization
    fig, ax = plt.subplots(
        figsize=(12, 8)
    )

    ax.imshow(image_rgb)

    # Ground truth CAN
    gx1, gy1, gx2, gy2 = can["xyxy"]

    rect_gt = plt.Rectangle(
        (gx1, gy1),
        gx2 - gx1,
        gy2 - gy1,
        fill=False,
        linewidth=3,
        linestyle="--"
    )

    ax.add_patch(rect_gt)

    ax.text(
        gx1,
        max(gy1 - 8, 5),
        "GROUND TRUTH: CAN",
        fontsize=11,
        fontweight="bold"
    )

    # Prediction
    if pred is not None:

        px1, py1, px2, py2 = pred["xyxy"]

        rect_pred = plt.Rectangle(
            (px1, py1),
            px2 - px1,
            py2 - py1,
            fill=False,
            linewidth=3
        )

        ax.add_patch(rect_pred)

        ax.text(
            px1,
            min(py2 + 18, height - 5),
            f"PRED: {pred_class} "
            f"conf={confidence:.3f} "
            f"IoU={can['best_iou']:.3f}",
            fontsize=11,
            fontweight="bold"
        )

    # Metadata
    area = can["area"]
    ar = can["aspect_ratio"]

    cx = can["xc"]
    cy = can["yc"]

    metadata = (
        f"Failure: {failure_type.upper()}\n"
        f"Image: {image_path.name}\n"
        f"CAN area: {area:.4f}\n"
        f"Aspect ratio: {ar:.3f}\n"
        f"Center: ({cx:.3f}, {cy:.3f})\n"
        f"Prediction: {pred_class}\n"
        f"Confidence: {confidence:.3f}\n"
        f"IoU: {can['best_iou']:.3f}"
    )

    ax.text(
        0.01,
        0.99,
        metadata,
        transform=ax.transAxes,
        verticalalignment="top",
        fontsize=10,
        bbox=dict(
            boxstyle="round",
            facecolor="white",
            alpha=0.85
        )
    )

    ax.axis("off")

    plt.tight_layout()

    # Safe filename
    safe_name = image_path.stem.replace(
        " ",
        "_"
    )

    if failure_type == "MISSED":

        output_path = MISSED_DIR / f"{safe_name}_MISSED.jpg"

    else:

        output_path = WRONG_DIR / f"{safe_name}_WRONG_CLASS.jpg"

    plt.savefig(
        output_path,
        dpi=150,
        bbox_inches="tight"
    )

    plt.close(fig)

    return {
        "filename": image_path.name,
        "failure_type": failure_type,
        "ground_truth_class": "CAN",
        "predicted_class": pred_class,
        "confidence": confidence,
        "iou": can["best_iou"],
        "area": can["area"],
        "aspect_ratio": can["aspect_ratio"],
        "center_x": can["xc"],
        "center_y": can["yc"],
        "output_image": str(output_path)
    }


# ======================================================================
# 9. PROCESS MISSED CAN CASES
# ======================================================================

print("\n" + "=" * 70)
print("PROCESSING MISSED CAN CASES")
print("=" * 70)

visual_results = []

for i, case in enumerate(
    missed_cases,
    start=1
):

    print(
        f"[MISSED {i}/{len(missed_cases)}] "
        f"{case['filename']}"
    )

    result = analyze_failure(
        case,
        "MISSED"
    )

    if result is not None:
        visual_results.append(result)


# ======================================================================
# 10. PROCESS WRONG-CLASS CASES
# ======================================================================

print("\n" + "=" * 70)
print("PROCESSING WRONG-CLASS CAN CASES")
print("=" * 70)

for i, case in enumerate(
    wrong_cases,
    start=1
):

    print(
        f"[WRONG {i}/{len(wrong_cases)}] "
        f"{case['filename']}"
    )

    result = analyze_failure(
        case,
        "WRONG_CLASS"
    )

    if result is not None:
        visual_results.append(result)


# ======================================================================
# 11. SAVE VISUAL ANALYSIS JSON
# ======================================================================

json_output = OUTPUT_ROOT / "section5_visual_failure_analysis.json"

with open(
    json_output,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        visual_results,
        f,
        indent=4
    )


# ======================================================================
# 12. FINAL SUMMARY
# ======================================================================

print("\n" + "=" * 70)
print("SECTION 5 — VISUAL INSPECTION COMPLETED")
print("=" * 70)

print(f"Missed CAN cases      : {len(missed_cases)}")
print(f"Wrong-class CAN cases : {len(wrong_cases)}")
print(f"Visualizations saved  : {len(visual_results)}")

print("\nOutput directories:")
print(f"Missed CAN      : {MISSED_DIR}")
print(f"Wrong-class CAN : {WRONG_DIR}")

print("\nAnalysis JSON:")
print(json_output)

print("\n" + "=" * 70)
print("[OK] SECTION 5 COMPLETED")
print("=" * 70)

EXPERIMENT 2 — CAN FAILURE ANALYSIS
SECTION 5 — VISUAL CAN FAILURE INSPECTION

[1] Configuration
----------------------------------------------------------------------
Model       : G:\EcoBotX_YOLO_training\experiment2_ecobotx_light-2\weights\best.pt
Test images : G:\EcoBotX_YOLO\images\test
Test labels : G:\EcoBotX_YOLO\labels\test
Section 4   : G:\EcoBotX_YOLO_training\experiment2_can_failure_analysis\can_failure_characterization.json
Output      : G:\EcoBotX_YOLO_training\experiment2_can_failure_analysis\section5_visual_inspection

[2] Checking required files...
[OK] Model found
[OK] Test image directory found
[OK] Test label directory found
[OK] Section 4 characterization found

[3] Loading Experiment 2 model...
[OK] Model loaded
Model classes: {0: 'BOTTLE', 1: 'CAN', 2: 'PAPER', 3: 'WRAPPER'}

[4] Loading Section 4 results...
[OK] Section 4 results loaded

[5] Extracting failure cases...
[INFO] Missed cases found      : 0
[INFO] Wrong-class cases found : 0

[INFO] Searching missed

In [1]:
# ======================================================================
# EXPERIMENT 2 — CAN FAILURE ANALYSIS
# SECTION 6 — CAN ROOT-CAUSE ANALYSIS
# ======================================================================

from pathlib import Path
import json
import statistics
from collections import Counter, defaultdict

from ultralytics import YOLO
from PIL import Image

print("=" * 70)
print("EXPERIMENT 2 — CAN FAILURE ANALYSIS")
print("SECTION 6 — CAN ROOT-CAUSE ANALYSIS")
print("=" * 70)


# ======================================================================
# 1. CONFIGURATION
# ======================================================================

MODEL_PATH = Path(
    r"G:\EcoBotX_YOLO_training\experiment2_ecobotx_light-2\weights\best.pt"
)

DATASET_YAML = Path(
    r"G:\EcoBotX_YOLO\dataset.yaml"
)

TEST_IMAGE_DIR = Path(
    r"G:\EcoBotX_YOLO\images\test"
)

TEST_LABEL_DIR = Path(
    r"G:\EcoBotX_YOLO\labels\test"
)

OUTPUT_DIR = Path(
    r"G:\EcoBotX_YOLO_training\experiment2_can_failure_analysis\section6_root_cause"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

JSON_OUTPUT = OUTPUT_DIR / "section6_root_cause_analysis.json"


CAN_CLASS_ID = 1

CLASS_NAMES = {
    0: "BOTTLE",
    1: "CAN",
    2: "PAPER",
    3: "WRAPPER"
}

IOU_THRESHOLD = 0.50
CONF_THRESHOLD = 0.25


# ======================================================================
# 2. CHECK PATHS
# ======================================================================

print("\n[1] Checking paths...")

assert MODEL_PATH.exists(), f"Model not found: {MODEL_PATH}"
assert DATASET_YAML.exists(), f"Dataset YAML not found: {DATASET_YAML}"
assert TEST_IMAGE_DIR.exists(), f"Test image directory not found: {TEST_IMAGE_DIR}"
assert TEST_LABEL_DIR.exists(), f"Test label directory not found: {TEST_LABEL_DIR}"

print("[OK] Experiment 2 model found")
print("[OK] Dataset YAML found")
print("[OK] Test image directory found")
print("[OK] Test label directory found")


# ======================================================================
# 3. LOAD MODEL
# ======================================================================

print("\n" + "=" * 70)
print("LOADING EXPERIMENT 2 MODEL")
print("=" * 70)

model = YOLO(str(MODEL_PATH))

print("[OK] Model loaded successfully")

print("\nModel classes:")
for k, v in CLASS_NAMES.items():
    print(f"  {k}: {v}")


# ======================================================================
# 4. HELPER FUNCTIONS
# ======================================================================

def calculate_iou(box1, box2):
    """
    Calculate IoU between two XYXY boxes.
    """

    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])

    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection_w = max(0.0, x2 - x1)
    intersection_h = max(0.0, y2 - y1)

    intersection = intersection_w * intersection_h

    area1 = max(0.0, box1[2] - box1[0]) * \
            max(0.0, box1[3] - box1[1])

    area2 = max(0.0, box2[2] - box2[0]) * \
            max(0.0, box2[3] - box2[1])

    union = area1 + area2 - intersection

    if union <= 0:
        return 0.0

    return intersection / union


def load_can_ground_truth(label_path):
    """
    Load all CAN ground-truth boxes from a YOLO label file.
    """

    cans = []

    with open(label_path, "r") as f:
        lines = f.readlines()

    for line in lines:

        parts = line.strip().split()

        if len(parts) != 5:
            continue

        cls = int(parts[0])

        if cls != CAN_CLASS_ID:
            continue

        xc = float(parts[1])
        yc = float(parts[2])
        w = float(parts[3])
        h = float(parts[4])

        cans.append({
            "class_id": cls,
            "xc": xc,
            "yc": yc,
            "w": w,
            "h": h,
            "area": w * h,
            "aspect_ratio": w / h if h > 0 else 0.0
        })

    return cans


def size_category(area):
    if area < 0.02:
        return "SMALL"
    elif area < 0.10:
        return "MEDIUM"
    else:
        return "LARGE"


def shape_category(aspect_ratio):

    if aspect_ratio > 2.0:
        return "VERY_WIDE"

    if aspect_ratio < 0.5:
        return "VERY_TALL"

    return "NORMAL"


def horizontal_position(xc):

    if xc < 0.33:
        return "LEFT"

    if xc > 0.66:
        return "RIGHT"

    return "CENTER"


def vertical_position(yc):

    if yc < 0.33:
        return "TOP"

    if yc > 0.66:
        return "BOTTOM"

    return "CENTER"


# ======================================================================
# 5. COLLECT CAN GROUND TRUTH
# ======================================================================

print("\n" + "=" * 70)
print("COLLECTING CAN GROUND-TRUTH")
print("=" * 70)

image_paths = sorted(
    list(TEST_IMAGE_DIR.glob("*.jpg")) +
    list(TEST_IMAGE_DIR.glob("*.jpeg")) +
    list(TEST_IMAGE_DIR.glob("*.png"))
)

print(f"Test images found: {len(image_paths)}")


# ======================================================================
# 6. RUN MODEL AND MATCH CAN PREDICTIONS
# ======================================================================

print("\n" + "=" * 70)
print("RUNNING EXPERIMENT 2")
print("=" * 70)

all_records = []

for index, image_path in enumerate(image_paths, start=1):

    label_path = TEST_LABEL_DIR / f"{image_path.stem}.txt"

    if not label_path.exists():
        continue

    can_gt = load_can_ground_truth(label_path)

    # Only images containing CAN are relevant here.
    if len(can_gt) == 0:
        continue

    image = Image.open(image_path)
    img_w, img_h = image.size

    result = model.predict(
        source=str(image_path),
        conf=CONF_THRESHOLD,
        verbose=False
    )[0]

    predictions = []

    if result.boxes is not None and len(result.boxes) > 0:

        boxes = result.boxes.xyxy.cpu().numpy()
        classes = result.boxes.cls.cpu().numpy()
        confidences = result.boxes.conf.cpu().numpy()

        for box, cls, conf in zip(
            boxes,
            classes,
            confidences
        ):

            predictions.append({
                "box": box.tolist(),
                "class_id": int(cls),
                "class_name": CLASS_NAMES.get(
                    int(cls),
                    str(int(cls))
                ),
                "confidence": float(conf)
            })

    # --------------------------------------------------------------
    # Match every CAN ground-truth box
    # --------------------------------------------------------------

    used_predictions = set()

    for gt_index, gt in enumerate(can_gt):

        gx = gt["xc"] * img_w
        gy = gt["yc"] * img_h
        gw = gt["w"] * img_w
        gh = gt["h"] * img_h

        gt_box = [
            gx - gw / 2,
            gy - gh / 2,
            gx + gw / 2,
            gy + gh / 2
        ]

        best_iou = 0.0
        best_prediction_index = None

        for pred_index, pred in enumerate(predictions):

            if pred_index in used_predictions:
                continue

            iou = calculate_iou(
                gt_box,
                pred["box"]
            )

            if iou > best_iou:
                best_iou = iou
                best_prediction_index = pred_index

        # ----------------------------------------------------------
        # Determine failure type
        # ----------------------------------------------------------

        if best_prediction_index is None:

            status = "MISSED"
            predicted_class = None
            confidence = 0.0

        else:

            pred = predictions[best_prediction_index]

            predicted_class = pred["class_name"]
            confidence = pred["confidence"]

            if best_iou >= IOU_THRESHOLD:

                used_predictions.add(best_prediction_index)

                if pred["class_id"] == CAN_CLASS_ID:

                    if confidence < 0.50:
                        status = "LOW_CONFIDENCE"
                    else:
                        status = "CORRECT"

                else:
                    status = "WRONG_CLASS"

            else:

                # Prediction exists but localization is insufficient.
                status = "MISSED"

        record = {
            "image": image_path.name,
            "status": status,

            "area": gt["area"],
            "width": gt["w"],
            "height": gt["h"],
            "aspect_ratio": gt["aspect_ratio"],

            "xc": gt["xc"],
            "yc": gt["yc"],

            "size_category": size_category(
                gt["area"]
            ),

            "shape_category": shape_category(
                gt["aspect_ratio"]
            ),

            "horizontal_position": horizontal_position(
                gt["xc"]
            ),

            "vertical_position": vertical_position(
                gt["yc"]
            ),

            "best_iou": float(best_iou),

            "predicted_class": predicted_class,
            "confidence": float(confidence)
        }

        all_records.append(record)


# ======================================================================
# 7. BASIC COUNTS
# ======================================================================

print("\n" + "=" * 70)
print("ROOT-CAUSE DATASET SUMMARY")
print("=" * 70)

total = len(all_records)

correct = sum(
    r["status"] == "CORRECT"
    for r in all_records
)

missed = sum(
    r["status"] == "MISSED"
    for r in all_records
)

wrong_class = sum(
    r["status"] == "WRONG_CLASS"
    for r in all_records
)

low_conf = sum(
    r["status"] == "LOW_CONFIDENCE"
    for r in all_records
)

failures = missed + wrong_class + low_conf

print(f"Total CAN annotations : {total}")
print(f"Correct               : {correct}")
print(f"Missed                : {missed}")
print(f"Wrong class           : {wrong_class}")
print(f"Low confidence        : {low_conf}")
print(f"Total failures        : {failures}")

if total > 0:
    print(
        f"Failure rate          : "
        f"{failures / total * 100:.2f}%"
    )

    print(
        f"CAN recall            : "
        f"{correct / total * 100:.2f}%"
    )


# ======================================================================
# 8. STATISTICAL COMPARISON
# ======================================================================

def values(records, key):

    return [
        r[key]
        for r in records
        if r[key] is not None
    ]


correct_records = [
    r for r in all_records
    if r["status"] == "CORRECT"
]

missed_records = [
    r for r in all_records
    if r["status"] == "MISSED"
]

wrong_records = [
    r for r in all_records
    if r["status"] == "WRONG_CLASS"
]

failed_records = [
    r for r in all_records
    if r["status"] != "CORRECT"
]


def print_statistics(title, records):

    print("\n" + "-" * 70)
    print(title)
    print("-" * 70)

    if not records:
        print("No records.")
        return

    for key, label in [
        ("area", "Area"),
        ("width", "Width"),
        ("height", "Height"),
        ("aspect_ratio", "Aspect Ratio"),
        ("xc", "X Center"),
        ("yc", "Y Center"),
        ("best_iou", "IoU"),
        ("confidence", "Confidence")
    ]:

        data = values(records, key)

        if not data:
            continue

        print(
            f"{label:<15} "
            f"Mean={statistics.mean(data):.4f} "
            f"Median={statistics.median(data):.4f} "
            f"Min={min(data):.4f} "
            f"Max={max(data):.4f}"
        )


print_statistics(
    "CORRECT CAN DETECTIONS",
    correct_records
)

print_statistics(
    "MISSED CAN DETECTIONS",
    missed_records
)

print_statistics(
    "WRONG-CLASS CAN DETECTIONS",
    wrong_records
)

print_statistics(
    "ALL FAILED CAN DETECTIONS",
    failed_records
)


# ======================================================================
# 9. FAILURE RATE BY OBJECT SIZE
# ======================================================================

print("\n" + "=" * 70)
print("FAILURE RATE BY CAN SIZE")
print("=" * 70)

size_stats = {}

for category in ["SMALL", "MEDIUM", "LARGE"]:

    subset = [
        r for r in all_records
        if r["size_category"] == category
    ]

    if not subset:
        continue

    fail = [
        r for r in subset
        if r["status"] != "CORRECT"
    ]

    rate = (
        len(fail) / len(subset) * 100
        if subset else 0
    )

    size_stats[category] = {
        "total": len(subset),
        "failures": len(fail),
        "failure_rate_percent": rate
    }

    print(
        f"{category:<10} "
        f"Total={len(subset):3d} "
        f"Failures={len(fail):3d} "
        f"Failure Rate={rate:.2f}%"
    )


# ======================================================================
# 10. FAILURE RATE BY SHAPE
# ======================================================================

print("\n" + "=" * 70)
print("FAILURE RATE BY CAN SHAPE")
print("=" * 70)

shape_stats = {}

for category in [
    "VERY_WIDE",
    "VERY_TALL",
    "NORMAL"
]:

    subset = [
        r for r in all_records
        if r["shape_category"] == category
    ]

    if not subset:
        continue

    fail = [
        r for r in subset
        if r["status"] != "CORRECT"
    ]

    rate = len(fail) / len(subset) * 100

    shape_stats[category] = {
        "total": len(subset),
        "failures": len(fail),
        "failure_rate_percent": rate
    }

    print(
        f"{category:<12} "
        f"Total={len(subset):3d} "
        f"Failures={len(fail):3d} "
        f"Failure Rate={rate:.2f}%"
    )


# ======================================================================
# 11. FAILURE RATE BY HORIZONTAL POSITION
# ======================================================================

print("\n" + "=" * 70)
print("FAILURE RATE BY HORIZONTAL POSITION")
print("=" * 70)

horizontal_stats = {}

for category in [
    "LEFT",
    "CENTER",
    "RIGHT"
]:

    subset = [
        r for r in all_records
        if r["horizontal_position"] == category
    ]

    if not subset:
        continue

    fail = [
        r for r in subset
        if r["status"] != "CORRECT"
    ]

    rate = len(fail) / len(subset) * 100

    horizontal_stats[category] = {
        "total": len(subset),
        "failures": len(fail),
        "failure_rate_percent": rate
    }

    print(
        f"{category:<10} "
        f"Total={len(subset):3d} "
        f"Failures={len(fail):3d} "
        f"Failure Rate={rate:.2f}%"
    )


# ======================================================================
# 12. FAILURE RATE BY VERTICAL POSITION
# ======================================================================

print("\n" + "=" * 70)
print("FAILURE RATE BY VERTICAL POSITION")
print("=" * 70)

vertical_stats = {}

for category in [
    "TOP",
    "CENTER",
    "BOTTOM"
]:

    subset = [
        r for r in all_records
        if r["vertical_position"] == category
    ]

    if not subset:
        continue

    fail = [
        r for r in subset
        if r["status"] != "CORRECT"
    ]

    rate = len(fail) / len(subset) * 100

    vertical_stats[category] = {
        "total": len(subset),
        "failures": len(fail),
        "failure_rate_percent": rate
    }

    print(
        f"{category:<10} "
        f"Total={len(subset):3d} "
        f"Failures={len(fail):3d} "
        f"Failure Rate={rate:.2f}%"
    )


# ======================================================================
# 13. WRONG-CLASS CONFUSION
# ======================================================================

print("\n" + "=" * 70)
print("CAN WRONG-CLASS CONFUSION")
print("=" * 70)

confusion = Counter(
    r["predicted_class"]
    for r in wrong_records
    if r["predicted_class"] is not None
)

if confusion:

    for predicted_class, count in confusion.most_common():

        percentage = (
            count / len(wrong_records) * 100
            if wrong_records else 0
        )

        print(
            f"CAN -> {predicted_class:<8} "
            f"{count:3d} "
            f"({percentage:.2f}%)"
        )

else:
    print("No wrong-class CAN detections.")


# ======================================================================
# 14. MOST PROBLEMATIC CAN CHARACTERISTICS
# ======================================================================

print("\n" + "=" * 70)
print("ROOT-CAUSE INDICATORS")
print("=" * 70)

if correct_records and failed_records:

    for key, label in [
        ("area", "Area"),
        ("width", "Width"),
        ("height", "Height"),
        ("aspect_ratio", "Aspect Ratio"),
        ("xc", "X Center"),
        ("yc", "Y Center")
    ]:

        correct_mean = statistics.mean(
            values(correct_records, key)
        )

        failed_mean = statistics.mean(
            values(failed_records, key)
        )

        difference = failed_mean - correct_mean

        print(
            f"{label:<15} "
            f"Correct={correct_mean:.4f} "
            f"Failed={failed_mean:.4f} "
            f"Difference={difference:+.4f}"
        )


# ======================================================================
# 15. FAILURE TYPE BREAKDOWN
# ======================================================================

print("\n" + "=" * 70)
print("FAILURE TYPE BREAKDOWN")
print("=" * 70)

failure_types = Counter(
    r["status"]
    for r in all_records
)

for status, count in failure_types.items():

    percentage = (
        count / total * 100
        if total else 0
    )

    print(
        f"{status:<18} "
        f"{count:3d} "
        f"({percentage:.2f}%)"
    )


# ======================================================================
# 16. SAVE COMPLETE JSON
# ======================================================================

analysis = {
    "experiment": "Experiment 2",
    "analysis": "CAN Root-Cause Analysis",

    "model_path": str(MODEL_PATH),
    "dataset_yaml": str(DATASET_YAML),
    "test_image_dir": str(TEST_IMAGE_DIR),
    "test_label_dir": str(TEST_LABEL_DIR),

    "can_class_id": CAN_CLASS_ID,

    "iou_threshold": IOU_THRESHOLD,
    "confidence_threshold": CONF_THRESHOLD,

    "summary": {
        "total_can_annotations": total,
        "correct": correct,
        "missed": missed,
        "wrong_class": wrong_class,
        "low_confidence": low_conf,
        "total_failures": failures,
        "failure_rate_percent":
            failures / total * 100 if total else 0,
        "recall_percent":
            correct / total * 100 if total else 0
    },

    "size_analysis": size_stats,
    "shape_analysis": shape_stats,
    "horizontal_position_analysis": horizontal_stats,
    "vertical_position_analysis": vertical_stats,

    "wrong_class_confusion":
        dict(confusion),

    "records": all_records
}


with open(JSON_OUTPUT, "w") as f:
    json.dump(
        analysis,
        f,
        indent=4
    )


# ======================================================================
# 17. FINAL SUMMARY
# ======================================================================

print("\n" + "=" * 70)
print("SECTION 6 — ROOT-CAUSE ANALYSIS COMPLETED")
print("=" * 70)

print(f"""
Total CAN annotations : {total}
Correct               : {correct}
Missed                : {missed}
Wrong class           : {wrong_class}
Low confidence        : {low_conf}
Total failures        : {failures}

Failure rate          : {
    failures / total * 100 if total else 0
:.2f}%

CAN recall            : {
    correct / total * 100 if total else 0
:.2f}%

Results saved to:
{JSON_OUTPUT}
""")

print("=" * 70)
print("[OK] SECTION 6 COMPLETED")
print("=" * 70)

EXPERIMENT 2 — CAN FAILURE ANALYSIS
SECTION 6 — CAN ROOT-CAUSE ANALYSIS

[1] Checking paths...
[OK] Experiment 2 model found
[OK] Dataset YAML found
[OK] Test image directory found
[OK] Test label directory found

LOADING EXPERIMENT 2 MODEL
[OK] Model loaded successfully

Model classes:
  0: BOTTLE
  1: CAN
  2: PAPER
  3: WRAPPER

COLLECTING CAN GROUND-TRUTH
Test images found: 1115

RUNNING EXPERIMENT 2

ROOT-CAUSE DATASET SUMMARY
Total CAN annotations : 228
Correct               : 180
Missed                : 22
Wrong class           : 10
Low confidence        : 16
Total failures        : 48
Failure rate          : 21.05%
CAN recall            : 78.95%

----------------------------------------------------------------------
CORRECT CAN DETECTIONS
----------------------------------------------------------------------
Area            Mean=0.2680 Median=0.2104 Min=0.0295 Max=0.8914
Width           Mean=0.5153 Median=0.5039 Min=0.1562 Max=0.9922
Height          Mean=0.4756 Median=0.4414 Mi

In [2]:
# ======================================================================
# EXPERIMENT 2 — CAN FAILURE ANALYSIS
# SECTION 7 — CORRECTED CAN FAILURE ANALYSIS
# ======================================================================

from pathlib import Path
import json
import statistics
from collections import Counter

from ultralytics import YOLO
from PIL import Image


print("=" * 70)
print("EXPERIMENT 2 — CAN FAILURE ANALYSIS")
print("SECTION 7 — CORRECTED CAN FAILURE ANALYSIS")
print("=" * 70)


# ======================================================================
# 1. CONFIGURATION
# ======================================================================

MODEL_PATH = Path(
    r"G:\EcoBotX_YOLO_training\experiment2_ecobotx_light-2\weights\best.pt"
)

DATASET_YAML = Path(
    r"G:\EcoBotX_YOLO\dataset.yaml"
)

TEST_IMAGE_DIR = Path(
    r"G:\EcoBotX_YOLO\images\test"
)

TEST_LABEL_DIR = Path(
    r"G:\EcoBotX_YOLO\labels\test"
)

OUTPUT_DIR = Path(
    r"G:\EcoBotX_YOLO_training\experiment2_can_failure_analysis\section7_corrected"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

JSON_OUTPUT = OUTPUT_DIR / "section7_corrected_can_analysis.json"


CAN_CLASS_ID = 1

CLASS_NAMES = {
    0: "BOTTLE",
    1: "CAN",
    2: "PAPER",
    3: "WRAPPER"
}

# Same basic thresholds used in the previous analysis
IOU_THRESHOLD = 0.50

# Confidence threshold used to identify low-confidence detections
LOW_CONFIDENCE_THRESHOLD = 0.50

# YOLO prediction threshold
PREDICTION_CONF_THRESHOLD = 0.25


# ======================================================================
# 2. CHECK PATHS
# ======================================================================

print("\n[1] Checking paths...")

assert MODEL_PATH.exists(), f"Model not found: {MODEL_PATH}"
assert DATASET_YAML.exists(), f"Dataset YAML not found: {DATASET_YAML}"
assert TEST_IMAGE_DIR.exists(), f"Test image directory not found: {TEST_IMAGE_DIR}"
assert TEST_LABEL_DIR.exists(), f"Test label directory not found: {TEST_LABEL_DIR}"

print("[OK] Experiment 2 model found")
print("[OK] Dataset YAML found")
print("[OK] Test image directory found")
print("[OK] Test label directory found")


# ======================================================================
# 3. LOAD MODEL
# ======================================================================

print("\n" + "=" * 70)
print("LOADING EXPERIMENT 2 MODEL")
print("=" * 70)

model = YOLO(str(MODEL_PATH))

print("[OK] Model loaded successfully")

print("\nModel classes:")

for class_id, name in CLASS_NAMES.items():
    print(f"  {class_id}: {name}")


# ======================================================================
# 4. HELPER FUNCTIONS
# ======================================================================

def calculate_iou(box1, box2):

    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])

    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection_w = max(0.0, x2 - x1)
    intersection_h = max(0.0, y2 - y1)

    intersection = intersection_w * intersection_h

    area1 = max(0.0, box1[2] - box1[0]) * \
            max(0.0, box1[3] - box1[1])

    area2 = max(0.0, box2[2] - box2[0]) * \
            max(0.0, box2[3] - box2[1])

    union = area1 + area2 - intersection

    if union <= 0:
        return 0.0

    return intersection / union


def load_can_ground_truth(label_path):

    cans = []

    with open(label_path, "r") as f:

        for line in f:

            parts = line.strip().split()

            if len(parts) != 5:
                continue

            class_id = int(parts[0])

            if class_id != CAN_CLASS_ID:
                continue

            xc = float(parts[1])
            yc = float(parts[2])
            w = float(parts[3])
            h = float(parts[4])

            cans.append({
                "class_id": class_id,
                "xc": xc,
                "yc": yc,
                "w": w,
                "h": h,
                "area": w * h,
                "aspect_ratio": w / h if h > 0 else 0.0
            })

    return cans


def size_category(area):

    if area < 0.02:
        return "SMALL"

    elif area < 0.10:
        return "MEDIUM"

    else:
        return "LARGE"


def shape_category(aspect_ratio):

    if aspect_ratio > 2.0:
        return "VERY_WIDE"

    elif aspect_ratio < 0.5:
        return "VERY_TALL"

    else:
        return "NORMAL"


def horizontal_position(x):

    if x < 0.33:
        return "LEFT"

    elif x > 0.66:
        return "RIGHT"

    else:
        return "CENTER"


def vertical_position(y):

    if y < 0.33:
        return "TOP"

    elif y > 0.66:
        return "BOTTOM"

    else:
        return "CENTER"


# ======================================================================
# 5. FIND TEST IMAGES
# ======================================================================

print("\n" + "=" * 70)
print("TEST DATASET")
print("=" * 70)

image_paths = sorted(
    list(TEST_IMAGE_DIR.glob("*.jpg")) +
    list(TEST_IMAGE_DIR.glob("*.jpeg")) +
    list(TEST_IMAGE_DIR.glob("*.png"))
)

print(f"Test images found: {len(image_paths)}")


# ======================================================================
# 6. RUN MODEL AND CLASSIFY EVERY CAN
# ======================================================================

print("\n" + "=" * 70)
print("RUNNING CORRECTED CAN ANALYSIS")
print("=" * 70)

records = []

for image_number, image_path in enumerate(
    image_paths,
    start=1
):

    label_path = TEST_LABEL_DIR / f"{image_path.stem}.txt"

    if not label_path.exists():
        continue

    can_ground_truth = load_can_ground_truth(label_path)

    # Ignore images without CAN
    if not can_ground_truth:
        continue

    image = Image.open(image_path)

    img_width, img_height = image.size

    result = model.predict(
        source=str(image_path),
        conf=PREDICTION_CONF_THRESHOLD,
        verbose=False
    )[0]

    predictions = []

    if result.boxes is not None and len(result.boxes) > 0:

        boxes = result.boxes.xyxy.cpu().numpy()
        classes = result.boxes.cls.cpu().numpy()
        confidences = result.boxes.conf.cpu().numpy()

        for box, cls, conf in zip(
            boxes,
            classes,
            confidences
        ):

            predictions.append({
                "box": box.tolist(),
                "class_id": int(cls),
                "class_name": CLASS_NAMES.get(
                    int(cls),
                    str(int(cls))
                ),
                "confidence": float(conf)
            })


    # --------------------------------------------------------------
    # MATCH EACH CAN GROUND-TRUTH BOX
    # --------------------------------------------------------------

    used_predictions = set()

    for gt_index, gt in enumerate(can_ground_truth):

        xc = gt["xc"] * img_width
        yc = gt["yc"] * img_height

        width = gt["w"] * img_width
        height = gt["h"] * img_height

        gt_box = [
            xc - width / 2,
            yc - height / 2,
            xc + width / 2,
            yc + height / 2
        ]

        best_iou = 0.0
        best_prediction_index = None

        for pred_index, prediction in enumerate(predictions):

            if pred_index in used_predictions:
                continue

            iou = calculate_iou(
                gt_box,
                prediction["box"]
            )

            if iou > best_iou:

                best_iou = iou
                best_prediction_index = pred_index


        # ----------------------------------------------------------
        # CORRECT CATEGORIZATION
        # ----------------------------------------------------------

        predicted_class = None
        confidence = 0.0

        if best_prediction_index is None:

            # No prediction at all
            status = "MISSED"

        else:

            prediction = predictions[
                best_prediction_index
            ]

            predicted_class = prediction["class_name"]
            confidence = prediction["confidence"]

            if best_iou >= IOU_THRESHOLD:

                used_predictions.add(
                    best_prediction_index
                )

                if prediction["class_id"] != CAN_CLASS_ID:

                    # Correct localization but wrong class
                    status = "WRONG_CLASS"

                else:

                    # Correctly localized and classified as CAN
                    if confidence < LOW_CONFIDENCE_THRESHOLD:

                        status = "CORRECT_LOW_CONFIDENCE"

                    else:

                        status = "CORRECT_HIGH_CONFIDENCE"

            else:

                # Prediction does not sufficiently overlap GT
                status = "MISSED"


        records.append({

            "image": image_path.name,

            "status": status,

            "area": gt["area"],
            "width": gt["w"],
            "height": gt["h"],
            "aspect_ratio": gt["aspect_ratio"],

            "xc": gt["xc"],
            "yc": gt["yc"],

            "size_category":
                size_category(gt["area"]),

            "shape_category":
                shape_category(
                    gt["aspect_ratio"]
                ),

            "horizontal_position":
                horizontal_position(
                    gt["xc"]
                ),

            "vertical_position":
                vertical_position(
                    gt["yc"]
                ),

            "best_iou":
                float(best_iou),

            "predicted_class":
                predicted_class,

            "confidence":
                float(confidence)
        })


# ======================================================================
# 7. FINAL CATEGORY COUNTS
# ======================================================================

print("\n" + "=" * 70)
print("SECTION 7 — CORRECTED CAN CLASSIFICATION")
print("=" * 70)

total = len(records)

correct_high = sum(
    r["status"] == "CORRECT_HIGH_CONFIDENCE"
    for r in records
)

correct_low = sum(
    r["status"] == "CORRECT_LOW_CONFIDENCE"
    for r in records
)

missed = sum(
    r["status"] == "MISSED"
    for r in records
)

wrong_class = sum(
    r["status"] == "WRONG_CLASS"
    for r in records
)

successful_detection = correct_high + correct_low

true_failures = missed + wrong_class


print(f"""
Total CAN annotations       : {total}

Correct CAN — high conf.    : {correct_high}
Correct CAN — low conf.     : {correct_low}

Missed CAN                  : {missed}
Wrong-class CAN             : {wrong_class}

Successful CAN detection    : {successful_detection}
True CAN failures           : {true_failures}
""")


# ======================================================================
# 8. CONSISTENCY CHECK
# ======================================================================

category_sum = (
    correct_high
    + correct_low
    + missed
    + wrong_class
)

print("=" * 70)
print("CONSISTENCY CHECK")
print("=" * 70)

print(
    f"Category sum : {category_sum}"
)

print(
    f"Total CANs   : {total}"
)

if category_sum == total:

    print(
        "[OK] Every CAN annotation belongs "
        "to exactly one category."
    )

else:

    print(
        "[ERROR] Category counts do not match "
        "total CAN annotations!"
    )


# ======================================================================
# 9. PERFORMANCE METRICS
# ======================================================================

true_detection_recall = (
    successful_detection / total * 100
    if total else 0
)

high_confidence_recall = (
    correct_high / total * 100
    if total else 0
)

low_confidence_rate = (
    correct_low / total * 100
    if total else 0
)

miss_rate = (
    missed / total * 100
    if total else 0
)

wrong_class_rate = (
    wrong_class / total * 100
    if total else 0
)

true_failure_rate = (
    true_failures / total * 100
    if total else 0
)


print("\n" + "=" * 70)
print("CORRECTED CAN PERFORMANCE")
print("=" * 70)

print(
    f"CAN detection recall       : "
    f"{true_detection_recall:.2f}%"
)

print(
    f"High-confidence recall     : "
    f"{high_confidence_recall:.2f}%"
)

print(
    f"Low-confidence rate        : "
    f"{low_confidence_rate:.2f}%"
)

print(
    f"Miss rate                  : "
    f"{miss_rate:.2f}%"
)

print(
    f"Wrong-class rate           : "
    f"{wrong_class_rate:.2f}%"
)

print(
    f"TRUE failure rate          : "
    f"{true_failure_rate:.2f}%"
)


# ======================================================================
# 10. CONFIDENCE ANALYSIS
# ======================================================================

print("\n" + "=" * 70)
print("CONFIDENCE ANALYSIS")
print("=" * 70)


def print_group_statistics(
    title,
    group
):

    print("\n" + "-" * 70)
    print(title)
    print("-" * 70)

    if not group:

        print("No records.")
        return

    areas = [
        r["area"]
        for r in group
    ]

    widths = [
        r["width"]
        for r in group
    ]

    heights = [
        r["height"]
        for r in group
    ]

    aspect_ratios = [
        r["aspect_ratio"]
        for r in group
    ]

    ious = [
        r["best_iou"]
        for r in group
    ]

    confidences = [
        r["confidence"]
        for r in group
        if r["confidence"] > 0
    ]

    print(
        f"Count             : {len(group)}"
    )

    print(
        f"Mean area         : "
        f"{statistics.mean(areas):.4f}"
    )

    print(
        f"Median area       : "
        f"{statistics.median(areas):.4f}"
    )

    print(
        f"Mean width        : "
        f"{statistics.mean(widths):.4f}"
    )

    print(
        f"Mean height       : "
        f"{statistics.mean(heights):.4f}"
    )

    print(
        f"Mean aspect ratio : "
        f"{statistics.mean(aspect_ratios):.4f}"
    )

    print(
        f"Mean IoU          : "
        f"{statistics.mean(ious):.4f}"
    )

    if confidences:

        print(
            f"Mean confidence   : "
            f"{statistics.mean(confidences):.4f}"
        )

        print(
            f"Median confidence : "
            f"{statistics.median(confidences):.4f}"
        )

        print(
            f"Minimum confidence: "
            f"{min(confidences):.4f}"
        )

        print(
            f"Maximum confidence: "
            f"{max(confidences):.4f}"
        )


correct_records = [
    r for r in records
    if r["status"] in [
        "CORRECT_HIGH_CONFIDENCE",
        "CORRECT_LOW_CONFIDENCE"
    ]
]

high_conf_records = [
    r for r in records
    if r["status"] == "CORRECT_HIGH_CONFIDENCE"
]

low_conf_records = [
    r for r in records
    if r["status"] == "CORRECT_LOW_CONFIDENCE"
]

missed_records = [
    r for r in records
    if r["status"] == "MISSED"
]

wrong_records = [
    r for r in records
    if r["status"] == "WRONG_CLASS"
]


print_group_statistics(
    "ALL CORRECT CAN DETECTIONS",
    correct_records
)

print_group_statistics(
    "HIGH-CONFIDENCE CORRECT CAN",
    high_conf_records
)

print_group_statistics(
    "LOW-CONFIDENCE CORRECT CAN",
    low_conf_records
)

print_group_statistics(
    "MISSED CAN",
    missed_records
)

print_group_statistics(
    "WRONG-CLASS CAN",
    wrong_records
)


# ======================================================================
# 11. FAILURE RATE BY SIZE
# ======================================================================

print("\n" + "=" * 70)
print("TRUE FAILURE RATE BY CAN SIZE")
print("=" * 70)

size_analysis = {}

for category in [
    "SMALL",
    "MEDIUM",
    "LARGE"
]:

    subset = [
        r for r in records
        if r["size_category"] == category
    ]

    if not subset:
        continue

    failures = [
        r for r in subset
        if r["status"] in [
            "MISSED",
            "WRONG_CLASS"
        ]
    ]

    low_conf = [
        r for r in subset
        if r["status"] == "CORRECT_LOW_CONFIDENCE"
    ]

    failure_rate = (
        len(failures) / len(subset) * 100
    )

    low_conf_rate = (
        len(low_conf) / len(subset) * 100
    )

    size_analysis[category] = {
        "total": len(subset),
        "true_failures": len(failures),
        "failure_rate_percent": failure_rate,
        "low_confidence": len(low_conf),
        "low_confidence_rate_percent": low_conf_rate
    }

    print(
        f"{category:<10} "
        f"Total={len(subset):3d} "
        f"Failures={len(failures):3d} "
        f"Failure Rate={failure_rate:.2f}% "
        f"LowConf={len(low_conf):3d} "
        f"LowConf Rate={low_conf_rate:.2f}%"
    )


# ======================================================================
# 12. FAILURE RATE BY SHAPE
# ======================================================================

print("\n" + "=" * 70)
print("TRUE FAILURE RATE BY CAN SHAPE")
print("=" * 70)

shape_analysis = {}

for category in [
    "VERY_WIDE",
    "VERY_TALL",
    "NORMAL"
]:

    subset = [
        r for r in records
        if r["shape_category"] == category
    ]

    if not subset:
        continue

    failures = [
        r for r in subset
        if r["status"] in [
            "MISSED",
            "WRONG_CLASS"
        ]
    ]

    rate = len(failures) / len(subset) * 100

    shape_analysis[category] = {
        "total": len(subset),
        "failures": len(failures),
        "failure_rate_percent": rate
    }

    print(
        f"{category:<12} "
        f"Total={len(subset):3d} "
        f"Failures={len(failures):3d} "
        f"Failure Rate={rate:.2f}%"
    )


# ======================================================================
# 13. FAILURE RATE BY HORIZONTAL POSITION
# ======================================================================

print("\n" + "=" * 70)
print("TRUE FAILURE RATE BY HORIZONTAL POSITION")
print("=" * 70)

horizontal_analysis = {}

for category in [
    "LEFT",
    "CENTER",
    "RIGHT"
]:

    subset = [
        r for r in records
        if r["horizontal_position"] == category
    ]

    if not subset:
        continue

    failures = [
        r for r in subset
        if r["status"] in [
            "MISSED",
            "WRONG_CLASS"
        ]
    ]

    rate = len(failures) / len(subset) * 100

    horizontal_analysis[category] = {
        "total": len(subset),
        "failures": len(failures),
        "failure_rate_percent": rate
    }

    print(
        f"{category:<10} "
        f"Total={len(subset):3d} "
        f"Failures={len(failures):3d} "
        f"Failure Rate={rate:.2f}%"
    )


# ======================================================================
# 14. FAILURE RATE BY VERTICAL POSITION
# ======================================================================

print("\n" + "=" * 70)
print("TRUE FAILURE RATE BY VERTICAL POSITION")
print("=" * 70)

vertical_analysis = {}

for category in [
    "TOP",
    "CENTER",
    "BOTTOM"
]:

    subset = [
        r for r in records
        if r["vertical_position"] == category
    ]

    if not subset:
        continue

    failures = [
        r for r in subset
        if r["status"] in [
            "MISSED",
            "WRONG_CLASS"
        ]
    ]

    rate = len(failures) / len(subset) * 100

    vertical_analysis[category] = {
        "total": len(subset),
        "failures": len(failures),
        "failure_rate_percent": rate
    }

    print(
        f"{category:<10} "
        f"Total={len(subset):3d} "
        f"Failures={len(failures):3d} "
        f"Failure Rate={rate:.2f}%"
    )


# ======================================================================
# 15. WRONG-CLASS CONFUSION
# ======================================================================

print("\n" + "=" * 70)
print("CAN WRONG-CLASS CONFUSION")
print("=" * 70)

confusion = Counter(
    r["predicted_class"]
    for r in wrong_records
    if r["predicted_class"] is not None
)

for predicted_class, count in confusion.most_common():

    percentage = (
        count / wrong_class * 100
        if wrong_class else 0
    )

    print(
        f"CAN -> {predicted_class:<10} "
        f"{count:3d} "
        f"({percentage:.2f}%)"
    )


# ======================================================================
# 16. MISSED CAN ANALYSIS
# ======================================================================

print("\n" + "=" * 70)
print("MISSED CAN ANALYSIS")
print("=" * 70)

if missed_records:

    missed_ious = [
        r["best_iou"]
        for r in missed_records
    ]

    missed_areas = [
        r["area"]
        for r in missed_records
    ]

    print(
        f"Missed CAN count      : "
        f"{len(missed_records)}"
    )

    print(
        f"Mean IoU              : "
        f"{statistics.mean(missed_ious):.4f}"
    )

    print(
        f"Median IoU            : "
        f"{statistics.median(missed_ious):.4f}"
    )

    print(
        f"Mean CAN area         : "
        f"{statistics.mean(missed_areas):.4f}"
    )

    print(
        f"Minimum IoU           : "
        f"{min(missed_ious):.4f}"
    )

    print(
        f"Maximum IoU           : "
        f"{max(missed_ious):.4f}"
    )

else:

    print("No missed CAN cases.")


# ======================================================================
# 17. WRONG-CLASS ANALYSIS
# ======================================================================

print("\n" + "=" * 70)
print("WRONG-CLASS CAN ANALYSIS")
print("=" * 70)

if wrong_records:

    wrong_ious = [
        r["best_iou"]
        for r in wrong_records
    ]

    wrong_areas = [
        r["area"]
        for r in wrong_records
    ]

    print(
        f"Wrong-class count     : "
        f"{len(wrong_records)}"
    )

    print(
        f"Mean IoU              : "
        f"{statistics.mean(wrong_ious):.4f}"
    )

    print(
        f"Mean CAN area         : "
        f"{statistics.mean(wrong_areas):.4f}"
    )

else:

    print("No wrong-class CAN cases.")


# ======================================================================
# 18. ROOT-CAUSE INTERPRETATION
# ======================================================================

print("\n" + "=" * 70)
print("PRELIMINARY ROOT-CAUSE INDICATORS")
print("=" * 70)

if missed_records and correct_records:

    correct_area_mean = statistics.mean(
        [r["area"] for r in correct_records]
    )

    missed_area_mean = statistics.mean(
        [r["area"] for r in missed_records]
    )

    correct_iou_mean = statistics.mean(
        [r["best_iou"] for r in correct_records]
    )

    missed_iou_mean = statistics.mean(
        [r["best_iou"] for r in missed_records]
    )

    print(
        f"Correct CAN mean area : "
        f"{correct_area_mean:.4f}"
    )

    print(
        f"Missed CAN mean area  : "
        f"{missed_area_mean:.4f}"
    )

    print(
        f"Area difference       : "
        f"{missed_area_mean - correct_area_mean:+.4f}"
    )

    print()

    print(
        f"Correct CAN mean IoU  : "
        f"{correct_iou_mean:.4f}"
    )

    print(
        f"Missed CAN mean IoU   : "
        f"{missed_iou_mean:.4f}"
    )

    print(
        f"IoU difference        : "
        f"{missed_iou_mean - correct_iou_mean:+.4f}"
    )


# ======================================================================
# 19. SAVE RESULTS
# ======================================================================

analysis = {

    "experiment":
        "Experiment 2",

    "section":
        "Section 7 — Corrected CAN Failure Analysis",

    "model_path":
        str(MODEL_PATH),

    "dataset_yaml":
        str(DATASET_YAML),

    "test_image_dir":
        str(TEST_IMAGE_DIR),

    "test_label_dir":
        str(TEST_LABEL_DIR),

    "thresholds": {

        "iou_threshold":
            IOU_THRESHOLD,

        "low_confidence_threshold":
            LOW_CONFIDENCE_THRESHOLD,

        "prediction_confidence_threshold":
            PREDICTION_CONF_THRESHOLD
    },

    "summary": {

        "total_can_annotations":
            total,

        "correct_high_confidence":
            correct_high,

        "correct_low_confidence":
            correct_low,

        "missed":
            missed,

        "wrong_class":
            wrong_class,

        "successful_detection":
            successful_detection,

        "true_failures":
            true_failures,

        "detection_recall_percent":
            true_detection_recall,

        "high_confidence_recall_percent":
            high_confidence_recall,

        "low_confidence_rate_percent":
            low_confidence_rate,

        "miss_rate_percent":
            miss_rate,

        "wrong_class_rate_percent":
            wrong_class_rate,

        "true_failure_rate_percent":
            true_failure_rate
    },

    "size_analysis":
        size_analysis,

    "shape_analysis":
        shape_analysis,

    "horizontal_position_analysis":
        horizontal_analysis,

    "vertical_position_analysis":
        vertical_analysis,

    "wrong_class_confusion":
        dict(confusion),

    "records":
        records
}


with open(
    JSON_OUTPUT,
    "w"
) as f:

    json.dump(
        analysis,
        f,
        indent=4
    )


# ======================================================================
# 20. FINAL SUMMARY
# ======================================================================

print("\n" + "=" * 70)
print("SECTION 7 — FINAL SUMMARY")
print("=" * 70)

print(f"""
Total CAN annotations       : {total}

Correct high-confidence     : {correct_high}
Correct low-confidence      : {correct_low}

Missed                      : {missed}
Wrong class                 : {wrong_class}

Successful detection        : {successful_detection}
TRUE failures               : {true_failures}

Detection recall            : {true_detection_recall:.2f}%
Low-confidence rate         : {low_confidence_rate:.2f}%
Miss rate                   : {miss_rate:.2f}%
Wrong-class rate            : {wrong_class_rate:.2f}%
TRUE failure rate           : {true_failure_rate:.2f}%

Category consistency        : {
    "PASSED"
    if category_sum == total
    else "FAILED"
}

Results saved to:
{JSON_OUTPUT}
""")

print("=" * 70)
print("[OK] SECTION 7 COMPLETED")
print("=" * 70)

EXPERIMENT 2 — CAN FAILURE ANALYSIS
SECTION 7 — CORRECTED CAN FAILURE ANALYSIS

[1] Checking paths...
[OK] Experiment 2 model found
[OK] Dataset YAML found
[OK] Test image directory found
[OK] Test label directory found

LOADING EXPERIMENT 2 MODEL
[OK] Model loaded successfully

Model classes:
  0: BOTTLE
  1: CAN
  2: PAPER
  3: WRAPPER

TEST DATASET
Test images found: 1115

RUNNING CORRECTED CAN ANALYSIS

SECTION 7 — CORRECTED CAN CLASSIFICATION

Total CAN annotations       : 228

Correct CAN — high conf.    : 180
Correct CAN — low conf.     : 16

Missed CAN                  : 22
Wrong-class CAN             : 10

Successful CAN detection    : 196
True CAN failures           : 32

CONSISTENCY CHECK
Category sum : 228
Total CANs   : 228
[OK] Every CAN annotation belongs to exactly one category.

CORRECTED CAN PERFORMANCE
CAN detection recall       : 85.96%
High-confidence recall     : 78.95%
Low-confidence rate        : 7.02%
Miss rate                  : 9.65%
Wrong-class rate         

In [3]:
# ======================================================================
# EXPERIMENT 2 — CAN FAILURE ANALYSIS
# SECTION 8 — VISUAL + STATISTICAL ROOT-CAUSE EVIDENCE
# ======================================================================

from pathlib import Path
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 70)
print("EXPERIMENT 2 — CAN FAILURE ANALYSIS")
print("SECTION 8 — VISUAL + STATISTICAL ROOT-CAUSE EVIDENCE")
print("=" * 70)


# ======================================================================
# 1. CONFIGURATION
# ======================================================================

MODEL_PATH = Path(
    r"G:\EcoBotX_YOLO_training\experiment2_ecobotx_light-2\weights\best.pt"
)

DATASET_ROOT = Path(r"G:\EcoBotX_YOLO")
TEST_IMAGE_DIR = DATASET_ROOT / "images" / "test"
TEST_LABEL_DIR = DATASET_ROOT / "labels" / "test"

OUTPUT_DIR = Path(
    r"G:\EcoBotX_YOLO_training"
    r"\\experiment2_can_failure_analysis"
    r"\\section8_root_cause_evidence"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CAN_CLASS_ID = 1

CLASS_NAMES = {
    0: "BOTTLE",
    1: "CAN",
    2: "PAPER",
    3: "WRAPPER"
}

print("\n[CONFIGURATION]")
print(f"Model       : {MODEL_PATH}")
print(f"Test images : {TEST_IMAGE_DIR}")
print(f"Test labels : {TEST_LABEL_DIR}")
print(f"Output      : {OUTPUT_DIR}")


# ======================================================================
# 2. CHECK PATHS
# ======================================================================

print("\n" + "=" * 70)
print("PATH VALIDATION")
print("=" * 70)

assert MODEL_PATH.exists(), f"Model not found: {MODEL_PATH}"
assert TEST_IMAGE_DIR.exists(), f"Test image directory not found: {TEST_IMAGE_DIR}"
assert TEST_LABEL_DIR.exists(), f"Test label directory not found: {TEST_LABEL_DIR}"

print("[OK] Model found")
print("[OK] Test image directory found")
print("[OK] Test label directory found")


# ======================================================================
# 3. LOAD SECTION 7 RESULTS
# ======================================================================

SECTION7_JSON = (
    Path(r"G:\EcoBotX_YOLO_training")
    / "experiment2_can_failure_analysis"
    / "section7_corrected"
    / "section7_corrected_can_analysis.json"
)

print("\n" + "=" * 70)
print("LOADING SECTION 7 RESULTS")
print("=" * 70)

if SECTION7_JSON.exists():
    with open(SECTION7_JSON, "r", encoding="utf-8") as f:
        section7_data = json.load(f)

    print("[OK] Section 7 JSON loaded")
    print(f"Source: {SECTION7_JSON}")
else:
    section7_data = None
    print("[WARNING] Section 7 JSON not found.")
    print("The analysis will continue using the test labels and model.")


# ======================================================================
# 4. LOAD YOLO MODEL
# ======================================================================

print("\n" + "=" * 70)
print("LOADING EXPERIMENT 2 MODEL")
print("=" * 70)

from ultralytics import YOLO

model = YOLO(str(MODEL_PATH))

print("[OK] Model loaded successfully")

print("\nModel classes:")
for k, v in model.names.items():
    print(f"  {k}: {v}")


# ======================================================================
# 5. READ CAN GROUND-TRUTH BOXES
# ======================================================================

print("\n" + "=" * 70)
print("READING CAN GROUND-TRUTH")
print("=" * 70)


def read_can_annotations(label_path):
    """
    Read CAN annotations from YOLO label file.

    Returns:
        list of dictionaries containing:
        class_id, xc, yc, width, height, area, aspect_ratio
    """

    cans = []

    if not label_path.exists():
        return cans

    with open(label_path, "r", encoding="utf-8") as f:

        for line in f:

            parts = line.strip().split()

            if len(parts) != 5:
                continue

            class_id = int(float(parts[0]))

            if class_id != CAN_CLASS_ID:
                continue

            xc = float(parts[1])
            yc = float(parts[2])
            w = float(parts[3])
            h = float(parts[4])

            area = w * h
            aspect_ratio = w / h if h > 0 else 0

            cans.append({
                "class_id": class_id,
                "xc": xc,
                "yc": yc,
                "width": w,
                "height": h,
                "area": area,
                "aspect_ratio": aspect_ratio
            })

    return cans


# ======================================================================
# 6. COLLECT ALL CAN GROUND-TRUTH DATA
# ======================================================================

all_can_data = []

image_files = sorted(
    [
        p for p in TEST_IMAGE_DIR.iterdir()
        if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
    ]
)

print(f"Test images found: {len(image_files)}")

for image_path in image_files:

    label_path = TEST_LABEL_DIR / f"{image_path.stem}.txt"

    annotations = read_can_annotations(label_path)

    for ann in annotations:

        record = {
            "image": image_path.name,
            "image_path": str(image_path),
            **ann
        }

        all_can_data.append(record)


gt_df = pd.DataFrame(all_can_data)

print(f"CAN annotations found: {len(gt_df)}")

assert len(gt_df) == 228, (
    f"Expected 228 CAN annotations, "
    f"but found {len(gt_df)}"
)

print("[OK] 228 CAN annotations confirmed")


# ======================================================================
# 7. CLASSIFY OBJECT SIZE
# ======================================================================

def classify_size(area):

    if area < 0.02:
        return "SMALL"

    elif area < 0.10:
        return "MEDIUM"

    else:
        return "LARGE"


def classify_shape(ar):

    if ar > 2.0:
        return "VERY_WIDE"

    elif ar < 0.5:
        return "VERY_TALL"

    else:
        return "NORMAL"


def classify_horizontal(x):

    if x < 0.33:
        return "LEFT"

    elif x < 0.67:
        return "CENTER"

    else:
        return "RIGHT"


def classify_vertical(y):

    if y < 0.33:
        return "TOP"

    elif y < 0.67:
        return "CENTER"

    else:
        return "BOTTOM"


gt_df["size_group"] = gt_df["area"].apply(classify_size)
gt_df["shape_group"] = gt_df["aspect_ratio"].apply(classify_shape)
gt_df["horizontal_group"] = gt_df["xc"].apply(classify_horizontal)
gt_df["vertical_group"] = gt_df["yc"].apply(classify_vertical)


# ======================================================================
# 8. RUN MODEL PREDICTIONS
# ======================================================================

print("\n" + "=" * 70)
print("RUNNING EXPERIMENT 2 ON TEST SET")
print("=" * 70)

results = model.predict(
    source=str(TEST_IMAGE_DIR),
    imgsz=640,
    conf=0.25,
    iou=0.50,
    device=0,
    verbose=False,
    save=False
)

print("[OK] Test inference completed")


# ======================================================================
# 9. HELPER — IoU
# ======================================================================

def calculate_iou(box1, box2):

    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])

    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection_w = max(0, x2 - x1)
    intersection_h = max(0, y2 - y1)

    intersection = intersection_w * intersection_h

    area1 = max(0, box1[2] - box1[0]) * max(0, box1[3] - box1[1])
    area2 = max(0, box2[2] - box2[0]) * max(0, box2[3] - box2[1])

    union = area1 + area2 - intersection

    if union <= 0:
        return 0.0

    return intersection / union


# ======================================================================
# 10. MATCH CAN GROUND-TRUTH TO MODEL PREDICTIONS
# ======================================================================

print("\n" + "=" * 70)
print("MATCHING CAN GROUND-TRUTH WITH PREDICTIONS")
print("=" * 70)


analysis_records = []

for _, gt in gt_df.iterrows():

    image_name = gt["image"]

    result = None

    for r in results:
        if Path(r.path).name == image_name:
            result = r
            break

    if result is None:
        continue

    image_width = result.orig_shape[1]
    image_height = result.orig_shape[0]

    # Ground-truth normalized box -> pixels
    gx = gt["xc"] * image_width
    gy = gt["yc"] * image_height
    gw = gt["width"] * image_width
    gh = gt["height"] * image_height

    gt_box = [
        gx - gw / 2,
        gy - gh / 2,
        gx + gw / 2,
        gy + gh / 2
    ]

    best_iou = 0.0
    best_class = None
    best_conf = 0.0

    if result.boxes is not None and len(result.boxes) > 0:

        boxes = result.boxes.xyxy.cpu().numpy()
        classes = result.boxes.cls.cpu().numpy()
        confidences = result.boxes.conf.cpu().numpy()

        for box, cls_id, conf in zip(
            boxes,
            classes,
            confidences
        ):

            iou = calculate_iou(gt_box, box)

            if iou > best_iou:

                best_iou = iou
                best_class = int(cls_id)
                best_conf = float(conf)

    # --------------------------------------------------------------
    # Determine failure type
    # --------------------------------------------------------------

    if best_iou >= 0.50 and best_class == CAN_CLASS_ID:

        if best_conf >= 0.50:
            status = "CORRECT_HIGH_CONF"
        else:
            status = "CORRECT_LOW_CONF"

    elif best_iou >= 0.50 and best_class is not None:

        status = "WRONG_CLASS"

    else:

        status = "MISSED"

    record = gt.to_dict()

    record.update({
        "best_iou": best_iou,
        "predicted_class": best_class,
        "predicted_class_name":
            CLASS_NAMES.get(best_class, "NONE"),
        "confidence": best_conf,
        "status": status
    })

    analysis_records.append(record)


analysis_df = pd.DataFrame(analysis_records)


# ======================================================================
# 11. SUMMARY
# ======================================================================

print("\n" + "=" * 70)
print("SECTION 8 — FAILURE SUMMARY")
print("=" * 70)

status_counts = analysis_df["status"].value_counts()

for status in [
    "CORRECT_HIGH_CONF",
    "CORRECT_LOW_CONF",
    "MISSED",
    "WRONG_CLASS"
]:

    count = int(status_counts.get(status, 0))

    print(f"{status:<22}: {count}")


# ======================================================================
# 12. FAILURE-ONLY DATA
# ======================================================================

failure_df = analysis_df[
    analysis_df["status"].isin(
        ["MISSED", "WRONG_CLASS"]
    )
].copy()

correct_df = analysis_df[
    analysis_df["status"].isin(
        ["CORRECT_HIGH_CONF", "CORRECT_LOW_CONF"]
    )
].copy()


# ======================================================================
# 13. FAILURE RATE BY OBJECT SIZE
# ======================================================================

print("\n" + "=" * 70)
print("FAILURE RATE BY OBJECT SIZE")
print("=" * 70)

size_rows = []

for group in ["SMALL", "MEDIUM", "LARGE"]:

    subset = analysis_df[
        analysis_df["size_group"] == group
    ]

    failures = subset[
        subset["status"].isin(["MISSED", "WRONG_CLASS"])
    ]

    total = len(subset)

    rate = (
        len(failures) / total * 100
        if total > 0 else 0
    )

    size_rows.append({
        "Group": group,
        "Total": total,
        "Failures": len(failures),
        "Failure_Rate_%": rate
    })

    print(
        f"{group:<10} "
        f"Total={total:<4} "
        f"Failures={len(failures):<4} "
        f"Failure Rate={rate:.2f}%"
    )


size_df = pd.DataFrame(size_rows)


# ======================================================================
# 14. FAILURE RATE BY SHAPE
# ======================================================================

print("\n" + "=" * 70)
print("FAILURE RATE BY SHAPE")
print("=" * 70)

shape_rows = []

for group in ["VERY_WIDE", "VERY_TALL", "NORMAL"]:

    subset = analysis_df[
        analysis_df["shape_group"] == group
    ]

    failures = subset[
        subset["status"].isin(["MISSED", "WRONG_CLASS"])
    ]

    total = len(subset)

    rate = (
        len(failures) / total * 100
        if total > 0 else 0
    )

    shape_rows.append({
        "Group": group,
        "Total": total,
        "Failures": len(failures),
        "Failure_Rate_%": rate
    })

    print(
        f"{group:<12} "
        f"Total={total:<4} "
        f"Failures={len(failures):<4} "
        f"Failure Rate={rate:.2f}%"
    )


shape_df = pd.DataFrame(shape_rows)


# ======================================================================
# 15. FAILURE RATE BY HORIZONTAL POSITION
# ======================================================================

print("\n" + "=" * 70)
print("FAILURE RATE BY HORIZONTAL POSITION")
print("=" * 70)

horizontal_rows = []

for group in ["LEFT", "CENTER", "RIGHT"]:

    subset = analysis_df[
        analysis_df["horizontal_group"] == group
    ]

    failures = subset[
        subset["status"].isin(["MISSED", "WRONG_CLASS"])
    ]

    total = len(subset)

    rate = (
        len(failures) / total * 100
        if total > 0 else 0
    )

    horizontal_rows.append({
        "Group": group,
        "Total": total,
        "Failures": len(failures),
        "Failure_Rate_%": rate
    })

    print(
        f"{group:<10} "
        f"Total={total:<4} "
        f"Failures={len(failures):<4} "
        f"Failure Rate={rate:.2f}%"
    )


horizontal_df = pd.DataFrame(horizontal_rows)


# ======================================================================
# 16. FAILURE RATE BY VERTICAL POSITION
# ======================================================================

print("\n" + "=" * 70)
print("FAILURE RATE BY VERTICAL POSITION")
print("=" * 70)

vertical_rows = []

for group in ["TOP", "CENTER", "BOTTOM"]:

    subset = analysis_df[
        analysis_df["vertical_group"] == group
    ]

    failures = subset[
        subset["status"].isin(["MISSED", "WRONG_CLASS"])
    ]

    total = len(subset)

    rate = (
        len(failures) / total * 100
        if total > 0 else 0
    )

    vertical_rows.append({
        "Group": group,
        "Total": total,
        "Failures": len(failures),
        "Failure_Rate_%": rate
    })

    print(
        f"{group:<10} "
        f"Total={total:<4} "
        f"Failures={len(failures):<4} "
        f"Failure Rate={rate:.2f}%"
    )


vertical_df = pd.DataFrame(vertical_rows)


# ======================================================================
# 17. COMPARE CORRECT VS FAILED CAN
# ======================================================================

print("\n" + "=" * 70)
print("CORRECT VS FAILED CAN COMPARISON")
print("=" * 70)

comparison_metrics = [
    "area",
    "width",
    "height",
    "aspect_ratio",
    "xc",
    "yc",
    "best_iou",
    "confidence"
]

comparison_rows = []

for metric in comparison_metrics:

    correct_values = correct_df[metric].dropna()
    failed_values = failure_df[metric].dropna()

    comparison_rows.append({
        "Metric": metric,
        "Correct_Mean": (
            correct_values.mean()
            if len(correct_values) else np.nan
        ),
        "Failed_Mean": (
            failed_values.mean()
            if len(failed_values) else np.nan
        ),
        "Difference": (
            failed_values.mean() - correct_values.mean()
            if len(correct_values) and len(failed_values)
            else np.nan
        )
    })


comparison_df = pd.DataFrame(comparison_rows)

print(comparison_df.to_string(index=False))


# ======================================================================
# 18. MISSED CAN ANALYSIS
# ======================================================================

missed_df = analysis_df[
    analysis_df["status"] == "MISSED"
].copy()

print("\n" + "=" * 70)
print("MISSED CAN ANALYSIS")
print("=" * 70)

if len(missed_df) > 0:

    print(f"Missed CANs: {len(missed_df)}")

    print(
        f"Mean area       : {missed_df['area'].mean():.4f}"
    )

    print(
        f"Mean width      : {missed_df['width'].mean():.4f}"
    )

    print(
        f"Mean height     : {missed_df['height'].mean():.4f}"
    )

    print(
        f"Mean IoU        : {missed_df['best_iou'].mean():.4f}"
    )

    print(
        f"Median IoU      : {missed_df['best_iou'].median():.4f}"
    )

    print(
        f"Mean confidence : {missed_df['confidence'].mean():.4f}"
    )


# ======================================================================
# 19. WRONG-CLASS ANALYSIS
# ======================================================================

wrong_df = analysis_df[
    analysis_df["status"] == "WRONG_CLASS"
].copy()

print("\n" + "=" * 70)
print("WRONG-CLASS CAN ANALYSIS")
print("=" * 70)

if len(wrong_df) > 0:

    print(f"Wrong-class CANs: {len(wrong_df)}")

    print("\nConfusion:")

    confusion = (
        wrong_df["predicted_class_name"]
        .value_counts()
    )

    for cls_name, count in confusion.items():

        percentage = count / len(wrong_df) * 100

        print(
            f"CAN -> {cls_name:<10} "
            f"{count:>3} "
            f"({percentage:.2f}%)"
        )

    print(
        f"\nMean IoU        : "
        f"{wrong_df['best_iou'].mean():.4f}"
    )

    print(
        f"Mean confidence : "
        f"{wrong_df['confidence'].mean():.4f}"
    )


# ======================================================================
# 20. CREATE VISUALIZATION — FAILURE RATE BY SIZE
# ======================================================================

print("\n" + "=" * 70)
print("GENERATING VISUALIZATIONS")
print("=" * 70)

plt.figure(figsize=(8, 5))

plt.bar(
    size_df["Group"],
    size_df["Failure_Rate_%"]
)

plt.xlabel("CAN Size")
plt.ylabel("Failure Rate (%)")
plt.title("Experiment 2 — CAN Failure Rate by Object Size")
plt.tight_layout()

size_plot = OUTPUT_DIR / "failure_rate_by_size.png"
plt.savefig(size_plot, dpi=300)
plt.close()

print(f"[OK] {size_plot}")


# ======================================================================
# 21. FAILURE RATE BY POSITION
# ======================================================================

plt.figure(figsize=(8, 5))

plt.bar(
    horizontal_df["Group"],
    horizontal_df["Failure_Rate_%"]
)

plt.xlabel("Horizontal Position")
plt.ylabel("Failure Rate (%)")
plt.title("Experiment 2 — CAN Failure Rate by Horizontal Position")
plt.tight_layout()

horizontal_plot = (
    OUTPUT_DIR / "failure_rate_horizontal_position.png"
)

plt.savefig(horizontal_plot, dpi=300)
plt.close()

print(f"[OK] {horizontal_plot}")


plt.figure(figsize=(8, 5))

plt.bar(
    vertical_df["Group"],
    vertical_df["Failure_Rate_%"]
)

plt.xlabel("Vertical Position")
plt.ylabel("Failure Rate (%)")
plt.title("Experiment 2 — CAN Failure Rate by Vertical Position")
plt.tight_layout()

vertical_plot = (
    OUTPUT_DIR / "failure_rate_vertical_position.png"
)

plt.savefig(vertical_plot, dpi=300)
plt.close()

print(f"[OK] {vertical_plot}")


# ======================================================================
# 22. AREA DISTRIBUTION
# ======================================================================

plt.figure(figsize=(8, 5))

plt.hist(
    correct_df["area"],
    bins=20,
    alpha=0.6,
    label="Correct"
)

plt.hist(
    failure_df["area"],
    bins=20,
    alpha=0.6,
    label="Failed"
)

plt.xlabel("Normalized Bounding-Box Area")
plt.ylabel("Number of CAN Annotations")
plt.title("CAN Bounding-Box Area — Correct vs Failed")
plt.legend()
plt.tight_layout()

area_plot = OUTPUT_DIR / "area_correct_vs_failed.png"

plt.savefig(area_plot, dpi=300)
plt.close()

print(f"[OK] {area_plot}")


# ======================================================================
# 23. IOU DISTRIBUTION
# ======================================================================

plt.figure(figsize=(8, 5))

plt.hist(
    correct_df["best_iou"],
    bins=20,
    alpha=0.7
)

plt.xlabel("Best IoU")
plt.ylabel("Number of CAN Annotations")
plt.title("CAN Localization Quality — Experiment 2")
plt.tight_layout()

iou_plot = OUTPUT_DIR / "iou_distribution.png"

plt.savefig(iou_plot, dpi=300)
plt.close()

print(f"[OK] {iou_plot}")


# ======================================================================
# 24. CONFIDENCE DISTRIBUTION
# ======================================================================

plt.figure(figsize=(8, 5))

plt.hist(
    correct_df["confidence"],
    bins=20,
    alpha=0.7
)

plt.xlabel("Prediction Confidence")
plt.ylabel("Number of CAN Annotations")
plt.title("CAN Prediction Confidence — Experiment 2")
plt.tight_layout()

confidence_plot = OUTPUT_DIR / "confidence_distribution.png"

plt.savefig(confidence_plot, dpi=300)
plt.close()

print(f"[OK] {confidence_plot}")


# ======================================================================
# 25. SAVE CSV FILES
# ======================================================================

analysis_csv = OUTPUT_DIR / "section8_can_analysis.csv"
size_csv = OUTPUT_DIR / "failure_by_size.csv"
shape_csv = OUTPUT_DIR / "failure_by_shape.csv"
horizontal_csv = OUTPUT_DIR / "failure_by_horizontal_position.csv"
vertical_csv = OUTPUT_DIR / "failure_by_vertical_position.csv"
comparison_csv = OUTPUT_DIR / "correct_vs_failed_comparison.csv"

analysis_df.to_csv(
    analysis_csv,
    index=False
)

size_df.to_csv(
    size_csv,
    index=False
)

shape_df.to_csv(
    shape_csv,
    index=False
)

horizontal_df.to_csv(
    horizontal_csv,
    index=False
)

vertical_df.to_csv(
    vertical_csv,
    index=False
)

comparison_df.to_csv(
    comparison_csv,
    index=False
)

print("\n[OK] CSV files saved")


# ======================================================================
# 26. SAVE COMPLETE JSON REPORT
# ======================================================================

report = {
    "experiment": "Experiment 2",
    "section": "Section 8 — Visual + Statistical Root-Cause Evidence",

    "model": str(MODEL_PATH),

    "dataset": {
        "root": str(DATASET_ROOT),
        "test_images": str(TEST_IMAGE_DIR),
        "test_labels": str(TEST_LABEL_DIR)
    },

    "total_can_annotations": int(len(analysis_df)),

    "status_counts": {
        str(k): int(v)
        for k, v in status_counts.items()
    },

    "failure_by_size":
        size_df.to_dict(orient="records"),

    "failure_by_shape":
        shape_df.to_dict(orient="records"),

    "failure_by_horizontal_position":
        horizontal_df.to_dict(orient="records"),

    "failure_by_vertical_position":
        vertical_df.to_dict(orient="records"),

    "correct_vs_failed":
        comparison_df.to_dict(orient="records"),

    "wrong_class_confusion": (
        wrong_df["predicted_class_name"]
        .value_counts()
        .to_dict()
    )
}

json_path = OUTPUT_DIR / "section8_root_cause_evidence.json"

with open(
    json_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        report,
        f,
        indent=4
    )


# ======================================================================
# 27. FINAL SUMMARY
# ======================================================================

print("\n" + "=" * 70)
print("SECTION 8 — FINAL SUMMARY")
print("=" * 70)

print(
    f"Total CAN annotations : "
    f"{len(analysis_df)}"
)

print(
    f"Correct high conf.   : "
    f"{int(status_counts.get('CORRECT_HIGH_CONF', 0))}"
)

print(
    f"Correct low conf.    : "
    f"{int(status_counts.get('CORRECT_LOW_CONF', 0))}"
)

print(
    f"Missed CAN           : "
    f"{int(status_counts.get('MISSED', 0))}"
)

print(
    f"Wrong class          : "
    f"{int(status_counts.get('WRONG_CLASS', 0))}"
)

true_failures = (
    int(status_counts.get("MISSED", 0))
    + int(status_counts.get("WRONG_CLASS", 0))
)

failure_rate = (
    true_failures / len(analysis_df) * 100
    if len(analysis_df) else 0
)

print(
    f"True failure rate    : "
    f"{failure_rate:.2f}%"
)

print("\nOutput directory:")
print(OUTPUT_DIR)

print("\nJSON report:")
print(json_path)

print("\n" + "=" * 70)
print("[OK] SECTION 8 COMPLETED")
print("=" * 70)

EXPERIMENT 2 — CAN FAILURE ANALYSIS
SECTION 8 — VISUAL + STATISTICAL ROOT-CAUSE EVIDENCE

[CONFIGURATION]
Model       : G:\EcoBotX_YOLO_training\experiment2_ecobotx_light-2\weights\best.pt
Test images : G:\EcoBotX_YOLO\images\test
Test labels : G:\EcoBotX_YOLO\labels\test
Output      : G:\EcoBotX_YOLO_training\experiment2_can_failure_analysis\section8_root_cause_evidence

PATH VALIDATION
[OK] Model found
[OK] Test image directory found
[OK] Test label directory found

LOADING SECTION 7 RESULTS
[OK] Section 7 JSON loaded
Source: G:\EcoBotX_YOLO_training\experiment2_can_failure_analysis\section7_corrected\section7_corrected_can_analysis.json

LOADING EXPERIMENT 2 MODEL
[OK] Model loaded successfully

Model classes:
  0: BOTTLE
  1: CAN
  2: PAPER
  3: WRAPPER

READING CAN GROUND-TRUTH
Test images found: 1115
CAN annotations found: 228
[OK] 228 CAN annotations confirmed

RUNNING EXPERIMENT 2 ON TEST SET
WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, whic

In [4]:

# ======================================================================
# EXPERIMENT 2 — CAN FAILURE ANALYSIS
# SECTION 9 — STATISTICAL SIGNIFICANCE + EFFECT-SIZE ANALYSIS
# ======================================================================

from pathlib import Path
import json
import math
import csv
import numpy as np

print("=" * 70)
print("EXPERIMENT 2 — CAN FAILURE ANALYSIS")
print("SECTION 9 — STATISTICAL SIGNIFICANCE + EFFECT-SIZE ANALYSIS")
print("=" * 70)


# ======================================================================
# 1. CONFIGURATION
# ======================================================================

MODEL_PATH = Path(
    r"G:\EcoBotX_YOLO_training\experiment2_ecobotx_light-2\weights\best.pt"
)

DATASET_YAML = Path(
    r"G:\EcoBotX_YOLO\dataset.yaml"
)

TEST_IMAGES = Path(
    r"G:\EcoBotX_YOLO\images\test"
)

TEST_LABELS = Path(
    r"G:\EcoBotX_YOLO\labels\test"
)

SECTION8_DIR = Path(
    r"G:\EcoBotX_YOLO_training\experiment2_can_failure_analysis"
    r"\section8_root_cause_evidence"
)

OUTPUT_DIR = Path(
    r"G:\EcoBotX_YOLO_training\experiment2_can_failure_analysis"
    r"\section9_statistical_analysis"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SECTION8_JSON = SECTION8_DIR / "section8_root_cause_evidence.json"

print("\n[1] Checking paths...")

assert MODEL_PATH.exists(), f"Model not found: {MODEL_PATH}"
assert DATASET_YAML.exists(), f"Dataset YAML not found: {DATASET_YAML}"
assert TEST_IMAGES.exists(), f"Test images not found: {TEST_IMAGES}"
assert TEST_LABELS.exists(), f"Test labels not found: {TEST_LABELS}"

print("[OK] Model found")
print("[OK] Dataset YAML found")
print("[OK] Test images found")
print("[OK] Test labels found")


# ======================================================================
# 2. IMPORTS
# ======================================================================

try:
    from scipy.stats import (
        mannwhitneyu,
        chi2_contingency,
        fisher_exact
    )

    SCIPY_AVAILABLE = True

except ImportError:
    SCIPY_AVAILABLE = False
    print("[WARNING] scipy is not installed.")


# ======================================================================
# 3. LOAD SECTION 8 DATA
# ======================================================================

print("\n" + "=" * 70)
print("LOADING SECTION 8 RESULTS")
print("=" * 70)

assert SECTION8_JSON.exists(), (
    f"Section 8 JSON not found:\n{SECTION8_JSON}"
)

with open(SECTION8_JSON, "r", encoding="utf-8") as f:
    section8_data = json.load(f)

print("[OK] Section 8 JSON loaded")
print(f"Source: {SECTION8_JSON}")


# ======================================================================
# 4. FIND PER-INSTANCE DATA
# ======================================================================

def recursively_find_records(obj):
    """
    Search nested JSON for a list of dictionaries containing
    per-instance CAN information.
    """
    candidates = []

    if isinstance(obj, list):
        if len(obj) > 0 and all(isinstance(x, dict) for x in obj):
            candidates.append(obj)

        for item in obj:
            candidates.extend(recursively_find_records(item))

    elif isinstance(obj, dict):
        for value in obj.values():
            candidates.extend(recursively_find_records(value))

    return candidates


record_candidates = recursively_find_records(section8_data)

records = None

for candidate in record_candidates:

    if len(candidate) < 20:
        continue

    keys = set()

    for item in candidate:
        keys.update(item.keys())

    useful_keys = {
        "area",
        "width",
        "height",
        "aspect_ratio",
        "xc",
        "yc",
        "confidence",
        "best_iou",
        "status",
        "category",
        "failure_type"
    }

    if len(keys.intersection(useful_keys)) >= 3:
        records = candidate
        break


# ======================================================================
# 5. IF SECTION 8 DOES NOT CONTAIN PER-INSTANCE DATA
#    RECONSTRUCT THE DATASET + MODEL PREDICTIONS
# ======================================================================

if records is None:

    print("\n[INFO] Section 8 JSON does not contain sufficient")
    print("       per-instance records.")
    print("[INFO] Reconstructing CAN-level analysis from dataset + model.")

    from ultralytics import YOLO
    from PIL import Image

    model = YOLO(str(MODEL_PATH))

    print("[OK] Model loaded")

    CAN_ID = 1

    image_files = sorted([
        p for p in TEST_IMAGES.iterdir()
        if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
    ])

    records = []

    def calculate_iou(box1, box2):

        x1 = max(box1[0], box2[0])
        y1 = max(box1[1], box2[1])
        x2 = min(box1[2], box2[2])
        y2 = min(box1[3], box2[3])

        inter_w = max(0.0, x2 - x1)
        inter_h = max(0.0, y2 - y1)

        intersection = inter_w * inter_h

        area1 = max(0.0, box1[2] - box1[0]) * \
                max(0.0, box1[3] - box1[1])

        area2 = max(0.0, box2[2] - box2[0]) * \
                max(0.0, box2[3] - box2[1])

        union = area1 + area2 - intersection

        if union <= 0:
            return 0.0

        return intersection / union


    print("\nRunning inference...")

    for index, image_path in enumerate(image_files, start=1):

        label_path = TEST_LABELS / f"{image_path.stem}.txt"

        if not label_path.exists():
            continue

        img = Image.open(image_path)
        img_w, img_h = img.size

        can_ground_truths = []

        with open(label_path, "r", encoding="utf-8") as f:

            for line in f:

                parts = line.strip().split()

                if len(parts) != 5:
                    continue

                cls, xc, yc, w, h = map(float, parts)

                if int(cls) != CAN_ID:
                    continue

                can_ground_truths.append({
                    "xc": xc,
                    "yc": yc,
                    "width": w,
                    "height": h,
                    "area": w * h,
                    "aspect_ratio": w / h if h > 0 else 0
                })


        if not can_ground_truths:
            continue


        result = model.predict(
            source=str(image_path),
            imgsz=640,
            conf=0.001,
            verbose=False
        )[0]


        predictions = []

        if result.boxes is not None:

            boxes = result.boxes.xyxy.cpu().numpy()
            classes = result.boxes.cls.cpu().numpy()
            confidences = result.boxes.conf.cpu().numpy()

            for box, cls, conf in zip(
                boxes,
                classes,
                confidences
            ):

                predictions.append({
                    "box": box.tolist(),
                    "class_id": int(cls),
                    "confidence": float(conf)
                })


        for gt in can_ground_truths:

            gt_box = [
                (gt["xc"] - gt["width"] / 2) * img_w,
                (gt["yc"] - gt["height"] / 2) * img_h,
                (gt["xc"] + gt["width"] / 2) * img_w,
                (gt["yc"] + gt["height"] / 2) * img_h
            ]


            best_iou = 0.0
            best_prediction = None

            for pred in predictions:

                pred_iou = calculate_iou(
                    gt_box,
                    pred["box"]
                )

                if pred_iou > best_iou:

                    best_iou = pred_iou
                    best_prediction = pred


            if best_prediction is None:

                status = "MISSED"
                confidence = 0.0

            elif best_iou < 0.5:

                status = "MISSED"
                confidence = best_prediction["confidence"]

            elif best_prediction["class_id"] != CAN_ID:

                status = "WRONG_CLASS"
                confidence = best_prediction["confidence"]

            else:

                confidence = best_prediction["confidence"]

                if confidence < 0.5:
                    status = "CORRECT_LOW_CONF"
                else:
                    status = "CORRECT_HIGH_CONF"


            if gt["area"] < 0.02:
                size = "SMALL"

            elif gt["area"] < 0.10:
                size = "MEDIUM"

            else:
                size = "LARGE"


            if gt["aspect_ratio"] > 2.0:
                shape = "VERY_WIDE"

            elif gt["aspect_ratio"] < 0.5:
                shape = "VERY_TALL"

            else:
                shape = "NORMAL"


            if gt["xc"] < 1 / 3:
                horizontal = "LEFT"

            elif gt["xc"] < 2 / 3:
                horizontal = "CENTER"

            else:
                horizontal = "RIGHT"


            if gt["yc"] < 1 / 3:
                vertical = "TOP"

            elif gt["yc"] < 2 / 3:
                vertical = "CENTER"

            else:
                vertical = "BOTTOM"


            records.append({
                "image": image_path.name,
                "area": gt["area"],
                "width": gt["width"],
                "height": gt["height"],
                "aspect_ratio": gt["aspect_ratio"],
                "xc": gt["xc"],
                "yc": gt["yc"],
                "size": size,
                "shape": shape,
                "horizontal": horizontal,
                "vertical": vertical,
                "best_iou": best_iou,
                "confidence": confidence,
                "status": status
            })


print(f"[OK] Per-instance CAN records available: {len(records)}")


# ======================================================================
# 6. NORMALIZE RECORD FORMAT
# ======================================================================

def get_value(record, possible_keys, default=None):

    for key in possible_keys:

        if key in record:
            return record[key]

    return default


normalized = []

for r in records:

    status = str(
        get_value(
            r,
            ["status", "category", "failure_type"],
            ""
        )
    ).upper()

    area = get_value(r, ["area"], None)
    width = get_value(r, ["width", "w"], None)
    height = get_value(r, ["height", "h"], None)
    aspect = get_value(
        r,
        ["aspect_ratio", "aspect"],
        None
    )

    xc = get_value(r, ["xc", "x_center"], None)
    yc = get_value(r, ["yc", "y_center"], None)

    iou = get_value(
        r,
        ["best_iou", "iou", "IoU"],
        None
    )

    confidence = get_value(
        r,
        ["confidence", "conf"],
        None
    )

    if None in [
        area,
        width,
        height,
        aspect,
        xc,
        yc,
        iou,
        confidence
    ]:
        continue

    normalized.append({
        "status": status,
        "area": float(area),
        "width": float(width),
        "height": float(height),
        "aspect_ratio": float(aspect),
        "xc": float(xc),
        "yc": float(yc),
        "best_iou": float(iou),
        "confidence": float(confidence)
    })


print(
    f"[OK] Valid statistical records: "
    f"{len(normalized)}"
)


# ======================================================================
# 7. CLASSIFY CORRECT VS FAILED
# ======================================================================

def is_correct(status):

    return status in [
        "CORRECT",
        "CORRECT_HIGH_CONF",
        "CORRECT_LOW_CONF",
        "SUCCESS"
    ]


correct = [
    r for r in normalized
    if is_correct(r["status"])
]

failed = [
    r for r in normalized
    if not is_correct(r["status"])
]


print("\n" + "=" * 70)
print("STATISTICAL DATASET")
print("=" * 70)

print(f"Total CAN records : {len(normalized)}")
print(f"Correct           : {len(correct)}")
print(f"Failed            : {len(failed)}")


# ======================================================================
# 8. MANN-WHITNEY U + EFFECT SIZE
# ======================================================================

def rank_biserial_effect(x, y):

    """
    Rank-biserial correlation based on Mann-Whitney U.

    Positive value means the first group tends to have
    larger values than the second group.
    """

    if len(x) == 0 or len(y) == 0:
        return None

    result = mannwhitneyu(
        x,
        y,
        alternative="two-sided"
    )

    U = result.statistic

    n1 = len(x)
    n2 = len(y)

    r_rb = (2 * U / (n1 * n2)) - 1

    return float(r_rb)


def effect_interpretation(effect):

    if effect is None:
        return "Not available"

    magnitude = abs(effect)

    if magnitude < 0.10:
        return "Negligible"

    elif magnitude < 0.30:
        return "Small"

    elif magnitude < 0.50:
        return "Moderate"

    else:
        return "Large"


continuous_variables = [
    "area",
    "width",
    "height",
    "aspect_ratio",
    "xc",
    "yc",
    "best_iou",
    "confidence"
]


statistical_results = {}


print("\n" + "=" * 70)
print("CONTINUOUS VARIABLE SIGNIFICANCE TESTS")
print("=" * 70)


for variable in continuous_variables:

    correct_values = [
        r[variable]
        for r in correct
    ]

    failed_values = [
        r[variable]
        for r in failed
    ]

    if len(correct_values) < 2 or len(failed_values) < 2:

        continue


    if SCIPY_AVAILABLE:

        test = mannwhitneyu(
            correct_values,
            failed_values,
            alternative="two-sided"
        )

        U = float(test.statistic)
        p = float(test.pvalue)

        effect = rank_biserial_effect(
            correct_values,
            failed_values
        )

    else:

        U = None
        p = None
        effect = None


    significant = (
        p < 0.05
        if p is not None
        else None
    )


    statistical_results[variable] = {
        "correct_mean": float(np.mean(correct_values)),
        "failed_mean": float(np.mean(failed_values)),
        "correct_median": float(np.median(correct_values)),
        "failed_median": float(np.median(failed_values)),
        "mann_whitney_U": U,
        "p_value": p,
        "significant_at_0_05": significant,
        "rank_biserial_effect": effect,
        "effect_interpretation": effect_interpretation(effect)
    }


    print(
        f"{variable:15s} "
        f"p={p:.6f} "
        f"effect={effect:.4f} "
        f"{effect_interpretation(effect)}"
        if p is not None and effect is not None
        else f"{variable:15s} statistical test unavailable"
    )


# ======================================================================
# 9. CATEGORICAL ANALYSIS
# ======================================================================

print("\n" + "=" * 70)
print("CATEGORICAL FAILURE ANALYSIS")
print("=" * 70)


def categorical_table(records, field):

    values = ["LEFT", "CENTER", "RIGHT"]

    if field == "vertical":
        values = ["TOP", "CENTER", "BOTTOM"]

    elif field == "size":
        values = ["SMALL", "MEDIUM", "LARGE"]

    elif field == "shape":
        values = [
            "VERY_WIDE",
            "VERY_TALL",
            "NORMAL"
        ]


    table = []

    for value in values:

        total = sum(
            1 for r in records
            if str(r.get(field, "")).upper() == value
        )

        failures = sum(
            1 for r in failed
            if str(r.get(field, "")).upper() == value
        )

        successes = total - failures

        table.append({
            "category": value,
            "total": total,
            "failed": failures,
            "correct": successes
        })

    return table


categorical_results = {}


# NOTE:
# The normalized records reconstructed above may not contain
# categorical fields if Section 8 only stored numerical values.
#
# Reconstruct them directly from the numerical values.

for r in normalized:

    area = r["area"]
    aspect = r["aspect_ratio"]
    xc = r["xc"]
    yc = r["yc"]


    if area < 0.02:
        r["size"] = "SMALL"
    elif area < 0.10:
        r["size"] = "MEDIUM"
    else:
        r["size"] = "LARGE"


    if aspect > 2.0:
        r["shape"] = "VERY_WIDE"
    elif aspect < 0.5:
        r["shape"] = "VERY_TALL"
    else:
        r["shape"] = "NORMAL"


    if xc < 1 / 3:
        r["horizontal"] = "LEFT"
    elif xc < 2 / 3:
        r["horizontal"] = "CENTER"
    else:
        r["horizontal"] = "RIGHT"


    if yc < 1 / 3:
        r["vertical"] = "TOP"
    elif yc < 2 / 3:
        r["vertical"] = "CENTER"
    else:
        r["vertical"] = "BOTTOM"


categorical_fields = [
    "size",
    "shape",
    "horizontal",
    "vertical"
]


for field in categorical_fields:

    table = categorical_table(
        normalized,
        field
    )

    categorical_results[field] = table

    print("\n" + field.upper())

    for row in table:

        rate = (
            row["failed"] / row["total"] * 100
            if row["total"] > 0
            else 0
        )

        print(
            f"{row['category']:12s} "
            f"Total={row['total']:3d} "
            f"Failed={row['failed']:3d} "
            f"FailureRate={rate:.2f}%"
        )


# ======================================================================
# 10. BUILD 2×K TABLES AND CHI-SQUARE TESTS
# ======================================================================

categorical_tests = {}


if SCIPY_AVAILABLE:

    for field, table in categorical_results.items():

        contingency = []

        correct_row = []
        failed_row = []

        for row in table:

            correct_row.append(row["correct"])
            failed_row.append(row["failed"])

        contingency = [
            correct_row,
            failed_row
        ]


        try:

            chi2, p, dof, expected = chi2_contingency(
                contingency
            )

            categorical_tests[field] = {
                "chi_square": float(chi2),
                "p_value": float(p),
                "degrees_of_freedom": int(dof),
                "significant_at_0_05": bool(p < 0.05),
                "contingency_table": contingency
            }

            print(
                f"\n{field.upper()} "
                f"Chi-square={chi2:.4f} "
                f"p={p:.6f}"
            )

        except Exception as e:

            categorical_tests[field] = {
                "error": str(e)
            }


# ======================================================================
# 11. IDENTIFY SIGNIFICANT VARIABLES
# ======================================================================

significant_continuous = []

for variable, result in statistical_results.items():

    p = result["p_value"]

    if p is not None and p < 0.05:

        significant_continuous.append({
            "variable": variable,
            "p_value": p,
            "effect": result["rank_biserial_effect"],
            "interpretation": result["effect_interpretation"]
        })


significant_categorical = []

for field, result in categorical_tests.items():

    p = result.get("p_value")

    if p is not None and p < 0.05:

        significant_categorical.append({
            "variable": field,
            "p_value": p
        })


# ======================================================================
# 12. PRINT SIGNIFICANCE SUMMARY
# ======================================================================

print("\n" + "=" * 70)
print("SIGNIFICANCE SUMMARY")
print("=" * 70)

if significant_continuous:

    print("\nSignificant continuous variables:")

    for item in significant_continuous:

        print(
            f"  {item['variable']:15s} "
            f"p={item['p_value']:.6f} "
            f"effect={item['effect']:.4f} "
            f"({item['interpretation']})"
        )

else:

    print("\nNo continuous variables reached p < 0.05.")


if significant_categorical:

    print("\nSignificant categorical variables:")

    for item in significant_categorical:

        print(
            f"  {item['variable']:15s} "
            f"p={item['p_value']:.6f}"
        )

else:

    print("\nNo categorical variables reached p < 0.05.")


# ======================================================================
# 13. CREATE PAPER-READY CSV
# ======================================================================

csv_path = OUTPUT_DIR / "section9_continuous_statistics.csv"

with open(
    csv_path,
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.writer(f)

    writer.writerow([
        "Variable",
        "Correct_Mean",
        "Failed_Mean",
        "Correct_Median",
        "Failed_Median",
        "Mann_Whitney_U",
        "P_Value",
        "Significant",
        "Rank_Biserial_Effect",
        "Effect_Interpretation"
    ])

    for variable, result in statistical_results.items():

        writer.writerow([
            variable,
            result["correct_mean"],
            result["failed_mean"],
            result["correct_median"],
            result["failed_median"],
            result["mann_whitney_U"],
            result["p_value"],
            result["significant_at_0_05"],
            result["rank_biserial_effect"],
            result["effect_interpretation"]
        ])


print(f"\n[OK] CSV saved:")
print(csv_path)


# ======================================================================
# 14. SAVE CATEGORICAL CSV
# ======================================================================

categorical_csv = OUTPUT_DIR / "section9_categorical_statistics.csv"

with open(
    categorical_csv,
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.writer(f)

    writer.writerow([
        "Variable",
        "Category",
        "Total",
        "Correct",
        "Failed",
        "Failure_Rate_Percent"
    ])

    for field, table in categorical_results.items():

        for row in table:

            rate = (
                row["failed"] / row["total"] * 100
                if row["total"] > 0
                else 0
            )

            writer.writerow([
                field,
                row["category"],
                row["total"],
                row["correct"],
                row["failed"],
                rate
            ])


print("[OK] Categorical CSV saved:")
print(categorical_csv)


# ======================================================================
# 15. SAVE COMPLETE JSON REPORT
# ======================================================================

report = {

    "experiment": "Experiment 2",

    "section": "Section 9 — Statistical Significance + Effect Size",

    "model": str(MODEL_PATH),

    "dataset": {
        "yaml": str(DATASET_YAML),
        "test_images": str(TEST_IMAGES),
        "test_labels": str(TEST_LABELS)
    },

    "sample": {
        "total_can": len(normalized),
        "correct": len(correct),
        "failed": len(failed),
        "failure_rate_percent":
            len(failed) / len(normalized) * 100
            if normalized else None
    },

    "continuous_analysis": statistical_results,

    "categorical_analysis": categorical_results,

    "categorical_tests": categorical_tests,

    "significant_continuous_variables":
        significant_continuous,

    "significant_categorical_variables":
        significant_categorical,

    "statistical_method": {
        "continuous":
            "Mann-Whitney U test, two-sided",
        "categorical":
            "Chi-square test of independence",
        "significance_level":
            0.05,
        "effect_size":
            "Rank-biserial correlation"
    }
}


json_path = OUTPUT_DIR / "section9_statistical_analysis.json"

with open(
    json_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        report,
        f,
        indent=4
    )


print("\n[OK] JSON report saved:")
print(json_path)


# ======================================================================
# 16. FINAL SUMMARY
# ======================================================================

print("\n" + "=" * 70)
print("SECTION 9 — FINAL SUMMARY")
print("=" * 70)

print(
    f"Total CAN records       : {len(normalized)}"
)

print(
    f"Correct CAN             : {len(correct)}"
)

print(
    f"Failed CAN              : {len(failed)}"
)

print(
    f"Failure rate            : "
    f"{len(failed) / len(normalized) * 100:.2f}%"
    if normalized
    else "N/A"
)

print(
    f"Significant continuous  : "
    f"{len(significant_continuous)}"
)

print(
    f"Significant categorical : "
    f"{len(significant_categorical)}"
)

print("\nOutput directory:")
print(OUTPUT_DIR)

print("\n" + "=" * 70)
print("[OK] SECTION 9 COMPLETED")
print("=" * 70)



EXPERIMENT 2 — CAN FAILURE ANALYSIS
SECTION 9 — STATISTICAL SIGNIFICANCE + EFFECT-SIZE ANALYSIS

[1] Checking paths...
[OK] Model found
[OK] Dataset YAML found
[OK] Test images found
[OK] Test labels found

LOADING SECTION 8 RESULTS
[OK] Section 8 JSON loaded
Source: G:\EcoBotX_YOLO_training\experiment2_can_failure_analysis\section8_root_cause_evidence\section8_root_cause_evidence.json

[INFO] Section 8 JSON does not contain sufficient
       per-instance records.
[INFO] Reconstructing CAN-level analysis from dataset + model.
[OK] Model loaded

Running inference...
[OK] Per-instance CAN records available: 228
[OK] Valid statistical records: 228

STATISTICAL DATASET
Total CAN records : 228
Correct           : 213
Failed            : 15

CONTINUOUS VARIABLE SIGNIFICANCE TESTS
area            p=0.035564 effect=-0.3252 Moderate
width           p=0.041420 effect=-0.3155 Moderate
height          p=0.129818 effect=-0.2344 Small
aspect_ratio    p=0.383909 effect=-0.1349 Small
xc              p